# 1. Solar filament instance segmentation at native resolution

This notebook trains the supplied **ModifiedUNet** on complete 2048 × 2048
grayscale H-alpha images. Two raw-logit channels predict foreground and
inter-instance separation. Individual COCO annotations, including overlaps and
disconnected regions, remain the ground truth for instance evaluation.

Attach the official competition dataset in Kaggle and enable two T4 GPUs (or one
CUDA GPU). No extra source uploads are required: readable cells below export the
entire runtime. Defaults use **4 CPU workers and 2 T4 GPUs** for full training.
Set `MODE = 'smoke'` in section 5 to run two native images per partition for one
epoch. Smoke checkpoints never produce a final test submission.

Sources: [competition](https://www.kaggle.com/competitions/filament-segmentation-2026),
[organizer evaluator](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook),
and the supplied local notebooks/model/loss. See repository docs for the source audit.


## 2. Evaluation and PQ mathematics

For independent masks $G_i,P_j$, $I_{ij}=|G_i\cap P_j|/|G_i\cup P_j|$.
The supplied organizer notebook marks **every pair with $I_{ij}>0.5$** as a true
positive. It does not perform Hungarian assignment. FP counts columns with no
qualifying match; FN counts unmatched rows. Aggregate over all annotator-image
records, then compute

$$PQ={\sum_{(i,j):I_{ij}>0.5} I_{ij}\over TP+\tfrac12 FP+\tfrac12 FN}.$$

The denominator-zero convention is 0. Predictions of a source image are scored
separately against each annotator. This matters when annotations overlap.
Dice/IoU distributions include all **positive-overlap pairs**; fragmentation and
merging use degree at IoU > 0. Validation uses original annotation masks, never
connected components of the foreground union. Validation selection is a global
instance PQ; loss is only a differentiable surrogate and training diagnostic.


## 3. Pipeline architecture

```text
COCO annotations ──> independent instance masks ─┬─> foreground union
                                               ├─> separation + overlap exclusion
                                               └─> original GT for PQ
Native image ─> normalize ─> augment ─> ModifiedUNet ─> [foreground, separation]
                            train with AMP/DDP          │
                     best validation PQ checkpoint <───┤
                                    validation cache ─> postprocess search
                              test full-image forward ─> instances ─> COCO RLE CSV
```

The default split keeps multiply annotated sources in training, as in the prior
full-image notebook. All records of a filename and all pixel-identical duplicates
stay together. No test annotations are read. A two-head pixel representation
cannot fully recover crossing/overlapping identities or arbitrary disconnected
membership; these remain explicit modeling limitations.


## 4. Dependencies and runtime checks

Kaggle supplies its CUDA-enabled PyTorch. Keep that build. The following cell
installs missing auxiliary packages, then exports the reusable source modules as
ordinary Python files. Each source cell is readable and can be edited directly.
The repository `requirements.txt` records a tested auxiliary package set.


In [ ]:
import importlib.util, os, sys, subprocess
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
# Each CPU worker gets one native-library thread; avoid nested thread pools.
for variable in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[variable] = "1"
missing = [
    package
    for module, package in [
        ("pycocotools", "pycocotools==2.0.11"),
        ("scipy", "scipy==1.18.1"),
        ("skimage", "scikit-image==0.26.0"),
        ("cv2", "opencv-python-headless==5.0.0.93"),
        ("pandas", "pandas==3.0.6"),
        ("PIL", "Pillow==12.3.0"),
        ("matplotlib", "matplotlib==3.11.2"),
    ]
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
SOURCE_DIR = Path.cwd() / "filament_runtime"
(SOURCE_DIR / "filament").mkdir(parents=True, exist_ok=True)
print("Exporting self-contained runtime to", SOURCE_DIR)

Runtime source: `unet.py`

In [ ]:
%%writefile filament_runtime/unet.py
"""Configurable version of the user's four-pool ModifiedUNet (raw logits)."""
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


class DoubleConv(nn.Module):
    """Two 3x3 convolutions; GroupNorm is stable with batch size one."""
    def __init__(self, in_channels: int, out_channels: int,
                 normalization: str = "group", activation: str = "relu"):
        super().__init__()
        def norm():
            if normalization == "group":
                groups = math.gcd(8, out_channels)
                if groups == out_channels and groups > 1: groups //= 2
                return nn.GroupNorm(groups, out_channels)
            if normalization == "batch": return nn.BatchNorm2d(out_channels)
            if normalization == "none": return nn.Identity()
            raise ValueError(f"Unknown normalization: {normalization}")
        def act():
            if activation == "relu": return nn.ReLU(inplace=False)
            if activation == "silu": return nn.SiLU(inplace=False)
            if activation == "leaky_relu": return nn.LeakyReLU(0.1, inplace=False)
            raise ValueError(f"Unknown activation: {activation}")
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), norm(), act(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), norm(), act())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)


class ModifiedUNet(nn.Module):
    """Original four-pool encoder and transpose-convolution decoder.

    For width f and 2048-square input: f@2048, 2f@1024, 4f@512,
    8f@256, 16f@128. Decoder reverses these through aligned skip connections.
    Channels 0/1 are foreground/separation; a third head is experimental.
    """
    def __init__(self, in_channels: int = 1, out_channels: int = 2,
                 init_features: int = 16, normalization: str = "group",
                 activation: str = "relu", checkpoint_blocks: bool = True):
        super().__init__()
        if min(in_channels, out_channels, init_features) < 1:
            raise ValueError("Channel counts must be positive")
        if checkpoint_blocks and normalization == "batch":
            raise ValueError("BatchNorm statistics update twice with checkpointing; disable it or use GroupNorm")
        self.checkpoint_blocks, self.in_channels = checkpoint_blocks, in_channels
        def conv(a, b): return DoubleConv(a, b, normalization, activation)
        f = init_features
        self.inc = conv(in_channels, f)
        for i in range(1, 5):
            setattr(self, f"down{i}", nn.Sequential(nn.MaxPool2d(2), conv(f*2**(i-1), f*2**i)))
        for i, scale in enumerate((8, 4, 2, 1), 1):
            setattr(self, f"up{i}", nn.ConvTranspose2d(f*scale*2, f*scale, 2, stride=2))
            setattr(self, f"conv_up{i}", conv(f*scale*2, f*scale))
        self.outc = nn.Conv2d(f, out_channels, 1)

    def _block(self, block, *args):
        if self.checkpoint_blocks and self.training and torch.is_grad_enabled():
            return checkpoint(block, *args, use_reentrant=False)
        return block(*args)

    @staticmethod
    def _match_and_concat(upsampled: torch.Tensor, bypass: torch.Tensor) -> torch.Tensor:
        dy, dx = bypass.shape[-2] - upsampled.shape[-2], bypass.shape[-1] - upsampled.shape[-1]
        upsampled = F.pad(upsampled, [dx//2, dx-dx//2, dy//2, dy-dy//2])
        return torch.cat([bypass, upsampled], dim=1)

    def _decode(self, x, skip, index):
        return getattr(self, f"conv_up{index}")(
            self._match_and_concat(getattr(self, f"up{index}")(x), skip))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 4 or x.shape[1] != self.in_channels or min(x.shape[-2:]) < 16:
            raise ValueError("Expected BCHW input with configured channels and H,W >= 16")
        skips = [self._block(self.inc, x)]
        for i in range(1, 5): skips.append(self._block(getattr(self, f"down{i}"), skips[-1]))
        x = skips[-1]
        for i, skip in enumerate(reversed(skips[:-1]), 1):
            def block(a, b, index=i): return self._decode(a, b, index)
            x = self._block(block, x, skip)
        return self.outc(x)


Runtime source: `filament_pq_proxy_loss.py`

In [ ]:
%%writefile filament_runtime/filament_pq_proxy_loss.py
import torch
import torch.nn as nn
import torch.nn.functional as F


def instance_boundaries(instance_ids: torch.Tensor) -> torch.Tensor:
    """Create a boundary target from a [B, H, W] integer instance-ID map.

    0 = background; 1, 2, ... = distinct filaments.
    Only contacts between two positive, distinct IDs are marked. Background
    contours and ambiguous overlap IDs (-1) are excluded. The production path
    uses precomputed proximity targets from independent annotation masks instead.
    Returns [B, 1, H, W] of type float32.
    """
    if instance_ids.ndim != 3:
        raise ValueError("instance_ids must have shape [B, H, W]")

    boundary = torch.zeros_like(instance_ids, dtype=torch.bool)

    a, b = instance_ids[:, 1:, :], instance_ids[:, :-1, :]
    vertical = (a != b) & (a > 0) & (b > 0)
    boundary[:, 1:, :] |= vertical
    boundary[:, :-1, :] |= vertical

    a, b = instance_ids[:, :, 1:], instance_ids[:, :, :-1]
    horizontal = (a != b) & (a > 0) & (b > 0)
    boundary[:, :, 1:] |= horizontal
    boundary[:, :, :-1] |= horizontal

    return boundary.unsqueeze(1).float()


def soft_erode(x: torch.Tensor) -> torch.Tensor:
    """Differentiable approximation to morphological erosion."""
    a = -F.max_pool2d(-x, kernel_size=(3, 1), stride=1, padding=(1, 0))
    b = -F.max_pool2d(-x, kernel_size=(1, 3), stride=1, padding=(0, 1))
    return torch.minimum(a, b)


def soft_open(x: torch.Tensor) -> torch.Tensor:
    return F.max_pool2d(soft_erode(x), kernel_size=3, stride=1, padding=1)


def soft_skeleton(x: torch.Tensor, iterations: int = 5) -> torch.Tensor:
    """Differentiable soft skeleton used by the soft-clDice surrogate."""
    skeleton = F.relu(x - soft_open(x))
    for _ in range(iterations):
        x = soft_erode(x)
        delta = F.relu(x - soft_open(x))
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


def soft_cldice_loss(
    probabilities: torch.Tensor, targets: torch.Tensor, iterations: int = 5
) -> torch.Tensor:
    predicted_centerline = soft_skeleton(probabilities, iterations)
    true_centerline = soft_skeleton(targets, iterations)

    axes = (1, 2, 3)
    eps = 1e-6
    precision = (
        (predicted_centerline * targets).sum(dim=axes) + eps
    ) / (predicted_centerline.sum(dim=axes) + eps)
    sensitivity = (
        (true_centerline * probabilities).sum(dim=axes) + eps
    ) / (true_centerline.sum(dim=axes) + eps)

    return (1 - (2 * precision * sensitivity + eps) / (
        precision + sensitivity + eps
    )).mean()


class FilamentPQProxyLoss(nn.Module):
    """Differentiable *proxy* for PQ-oriented filament segmentation.

    Inputs
    ------
    logits : [B, 2, H, W]
        channel 0 = foreground logits
        channel 1 = instance-boundary logits
    instance_ids : [B, H, W]
        0 = background; positive integers identify distinct filaments

    This does NOT directly optimize official Panoptic Quality.
    """
    def __init__(
        self,
        bce_weight: float = 0.35,
        dice_weight: float = 0.40,
        boundary_weight: float = 0.25,
        cldice_weight: float = 0.0,
        boundary_pos_weight: float = 5.0,
        skeleton_iterations: int = 5,
    ):
        super().__init__()
        if min(bce_weight, dice_weight, boundary_weight, cldice_weight) < 0:
            raise ValueError("Loss weights cannot be negative")
        if bce_weight + dice_weight + boundary_weight + cldice_weight == 0:
            raise ValueError("At least one loss weight must be positive")
        if skeleton_iterations < 0:
            raise ValueError("skeleton_iterations cannot be negative")

        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.boundary_weight = boundary_weight
        self.cldice_weight = cldice_weight
        self.skeleton_iterations = skeleton_iterations
        if boundary_pos_weight <= 0:
            raise ValueError("boundary_pos_weight must be positive")
        self.register_buffer(
            "boundary_pos_weight", torch.tensor(float(boundary_pos_weight))
        )

    def forward(self, logits: torch.Tensor, targets):
        """Accept explicit foreground/separation/valid BCHW maps, or legacy IDs.

        The explicit maps preserve overlap foreground; integer-ID compatibility
        cannot represent overlapping masks and is for old callers only.
        """
        if logits.ndim != 4 or logits.shape[1] != 2:
            raise ValueError("Expected logits [B, 2, H, W]")
        if isinstance(targets, torch.Tensor):
            if targets.ndim != 3:
                raise ValueError("Expected legacy instance IDs BHW")
            foreground = (targets != 0).unsqueeze(1).float()
            boundaries = instance_boundaries(targets)
            valid = (targets >= 0).unsqueeze(1).float()
        else:
            foreground, boundaries, valid = (targets[k].float() for k in
                ("foreground", "separation", "separation_valid"))
        if any(t.shape != logits[:, :1].shape for t in (foreground, boundaries, valid)):
            raise ValueError("All loss target maps must be B1HW aligned with logits")
        # Explicitly disable ambient autocast: sums and BCE remain FP32.
        with torch.autocast(device_type=logits.device.type, enabled=False):
            mask_logits, boundary_logits = logits[:, :1].float(), logits[:, 1:2].float()
            probabilities = torch.sigmoid(mask_logits)
            bce = F.binary_cross_entropy_with_logits(mask_logits, foreground)
            axes = (1, 2, 3)
            intersection = (probabilities * foreground).sum(dim=axes)
            denominator = probabilities.sum(dim=axes) + foreground.sum(dim=axes)
            dice = (1 - (2 * intersection + 1.0) / (denominator + 1.0)).mean()
            boundary_pixels = F.binary_cross_entropy_with_logits(
                boundary_logits, boundaries, pos_weight=self.boundary_pos_weight, reduction="none")
            # Per-image reduction makes uneven accumulation windows equivalent
            # to the mean of their samples even when overlap exclusions vary.
            boundary_bce = ((boundary_pixels * valid).sum(axes) / valid.sum(axes).clamp_min(1)).mean()
            cldice = soft_cldice_loss(probabilities, foreground, iterations=self.skeleton_iterations
            ) if self.cldice_weight > 0 else mask_logits.new_zeros(())
            total = (
                self.bce_weight * bce + self.dice_weight * dice
                + self.boundary_weight * boundary_bce + self.cldice_weight * cldice
            )
        return {
            "loss": total,
            "bce": bce.detach(),
            "dice": dice.detach(),
            "boundary": boundary_bce.detach(),
            "cldice": cldice.detach(),
        }


In [ ]:
%%writefile -a filament_runtime/filament_pq_proxy_loss.py
if __name__ == "__main__":
    # Small smoke test, including a 2-instance target and backpropagation.
    model_output = torch.randn(2, 2, 63, 65, requires_grad=True)
    instances = torch.zeros(2, 63, 65, dtype=torch.long)
    instances[0, 5:35, 8:12] = 1
    instances[0, 35:58, 8:12] = 2  # Touching filament instances
    instances[1, 12:48, 20:25] = 1

    criterion = FilamentPQProxyLoss(cldice_weight=0.10)
    values = criterion(model_output, instances)
    values["loss"].backward()

    assert torch.isfinite(values["loss"])
    assert model_output.grad is not None
    assert torch.isfinite(model_output.grad).all()
    print({k: round(v.item(), 4) for k, v in values.items()})
    print("Backward pass: OK")


Runtime source: `filament/__init__.py`

In [ ]:
%%writefile filament_runtime/filament/__init__.py
"""Native-resolution solar filament instance segmentation pipeline."""
__version__ = "1.0.0"


Runtime source: `filament/cli.py`

In [ ]:
%%writefile filament_runtime/filament/cli.py
"""python -m filament.cli prepare|train|worker|infer|smoke."""
import argparse
import json
import os
import time
from pathlib import Path
from .config import Config
from .io import atomic_json,file_hash


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument("command",choices=["prepare","train","worker","evaluate","infer","smoke"])
    parser.add_argument("--config"); parser.add_argument("--runtime")
    parser.add_argument("--gpus",type=int); parser.add_argument("--output",default="filament_outputs")
    args=parser.parse_args()
    if args.command=="worker":
        started=time.perf_counter()
        print(f"[rank {os.environ.get('RANK',0)}] Worker started; importing PyTorch and training runtime...",flush=True)
    import torch
    import numpy as np
    if args.command=="worker":
        from .training import train
        print(f"[rank {os.environ.get('RANK',0)}] Runtime imports ready ({time.perf_counter()-started:.1f}s)",flush=True)
        runtime=json.loads(Path(args.runtime).read_text(encoding="utf8"))
        train(Config(**runtime["config"]),runtime["train_records"],runtime["val_records"],runtime["split_fingerprint"])
        return
    config=Config(**json.loads(Path(args.config).read_text(encoding="utf8"))) if args.config else Config(output_dir=args.output)
    if args.command=="smoke":
        config.smoke=True; config.epochs=1; config.microbatch=1
        config.model["init_features"]=8
    if args.command in ("prepare","train","smoke"):
        from .workflow import prepare_experiment,launch_training
        runtime=prepare_experiment(config)
        if args.command!="prepare": launch_training(runtime,args.gpus)
    elif args.command=="evaluate":
        from .training import load_checkpoint
        from .model import build_model
        from .inference import cache_validation_predictions,search_postprocessing,make_predictor
        from .visualization import plot_training,plot_distributions,plot_error_gallery
        import matplotlib.pyplot as plt
        output=Path(config.output_dir)
        runtime=json.loads((output/"prepared.json").read_text(encoding="utf8"))
        checkpoint=output/"checkpoints/best.pt"
        state=load_checkpoint(checkpoint); effective=Config(**state["config"])
        model=build_model(effective.model)
        model.load_state_dict(state["model"]); del state
        device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu"); model=model.to(device)
        if effective.channels_last: model.to(memory_format=torch.channels_last)
        predictor=make_predictor(model,effective,device,args.gpus)
        manifest=cache_validation_predictions(model,runtime["val_records"],output/"validation_predictions",device,
            dict(checkpoint_sha256=file_hash(checkpoint),normalization=effective.normalization,split_fingerprint=runtime["split_fingerprint"]),
            workers=effective.workers,predictor=predictor,
            amp=effective.amp,channels_last=effective.channels_last,normalization=effective.normalization)
        grid=dict(method=["components","boundary","watershed"],foreground_threshold=[.4,.5,.6],min_area=[10,20])
        if effective.smoke: grid=dict(method=["components","boundary"],foreground_threshold=[.5])
        best,frame=search_postprocessing(runtime["val_records"],manifest,grid,output/"search",effective.postprocess,workers=effective.workers)
        details=json.loads((output/"search/best_validation_details.json").read_text(encoding="utf8"))
        plt.close(plot_training(output/"logs/history.csv",output/"figures"))
        plt.close(plot_distributions(details,output/"figures/overlap_distributions.png"))
        plot_error_gallery(runtime["val_records"],manifest,details,best["postprocess"],output/"figures/errors")
        print(best)
    elif args.command=="infer":
        from .data import find_dataset_root,discover_test_images,read_grayscale
        from .training import load_checkpoint
        from .model import build_model
        from .inference import make_predictor
        from .postprocess import extract_instances
        from .submission import generate_submission
        output=Path(config.output_dir)
        checkpoint=output/"checkpoints/best.pt"
        state=load_checkpoint(checkpoint); config=Config(**state["config"])
        if config.smoke: raise ValueError("A smoke checkpoint cannot generate a final test submission")
        device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        model=build_model(config.model).to(device); model.load_state_dict(state["model"])
        del state
        if config.channels_last: model.to(memory_format=torch.channels_last)
        predictor=make_predictor(model,config,device,args.gpus)
        selection=json.loads((output/"search/best_postprocessing.json").read_text(encoding="utf8"))
        if selection["prediction_provenance"]["checkpoint_sha256"]!=file_hash(checkpoint):
            raise ValueError("Postprocessing selection belongs to a different checkpoint")
        post=selection["postprocess"]
        paths=discover_test_images(find_dataset_root(config.data_root),workers=config.workers)
        def predict(path):
            p=predictor(path)
            return extract_instances(p.astype(np.float16),post)[1]
        report=generate_submission(paths,predict,output/"submission.csv",output/"inference_journal",
            dict(checkpoint_sha256=file_hash(checkpoint),postprocess=post,normalization=config.normalization),workers=config.workers)
        print(report)


if __name__=="__main__": main()


Runtime source: `filament/config.py`

In [ ]:
%%writefile filament_runtime/filament/config.py
"""Serializable central experiment configuration."""
from dataclasses import asdict, dataclass, field


@dataclass
class Config:
    data_root: str = ""
    output_dir: str = "filament_outputs"
    image_size: int = 2048  # Only synthetic APIs may change this.
    seed: int = 10
    val_fraction: float = 0.2
    multi_annotator_train_only: bool = True
    model: dict = field(default_factory=lambda: dict(name="modified", in_channels=1,
        out_channels=2, init_features=16, normalization="group", activation="relu", checkpoint_blocks=True))
    loss: dict = field(default_factory=lambda: dict(bce_weight=.35, dice_weight=.40,
        boundary_weight=.25, cldice_weight=0., boundary_pos_weight=5., skeleton_iterations=5))
    separation_radius: int = 1
    centerline: bool = False
    normalization: str = "unit"
    augment: dict = field(default_factory=lambda: dict(flips=True, rotate90=True,
        brightness=.05, contrast=.10, gamma=.10, noise=0.))
    epochs: int = 50
    patience: int = 8
    pq_every: int = 1
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    scheduler: str = "plateau"
    accumulation_steps: int = 4
    microbatch: int | str = "auto"
    batch_candidates: list = field(default_factory=lambda: [1, 2])  # Dual-T4 log: 2 fits, 4 OOMs.
    width_fallbacks: list = field(default_factory=lambda: [8])
    memory_fraction: float = .85
    grad_clip: float = 1.
    amp: bool = True
    channels_last: bool = True
    workers: int = 4  # Total CPU worker budget, split across GPU ranks.
    prefetch_factor: int = 2
    log_every: int = 20
    resume: str = ""
    deterministic: bool = True
    smoke: bool = False
    smoke_records: int = 2
    postprocess: dict = field(default_factory=lambda: dict(method="boundary", foreground_threshold=.5,
        boundary_threshold=.5, min_area=20, seed_min_area=3, seed_distance=5,
        boundary_weight=2., closing_radius=0, reconnect_distance=0))

    def validate(self):
        if self.image_size != 2048: raise ValueError("Production requires native 2048x2048")
        if self.model.get("out_channels") != 2: raise ValueError("Production trains exactly two output channels")
        if self.normalization not in ("unit", "robust"): raise ValueError("Unknown normalization")
        if min(self.epochs, self.patience, self.pq_every, self.accumulation_steps, self.prefetch_factor, self.log_every) < 1:
            raise ValueError("Epoch, evaluation, accumulation and prefetch counts must be positive")
        if self.workers < 0 or not 0 < self.memory_fraction < 1 or not 0 < self.val_fraction < 1:
            raise ValueError("Invalid workers, VRAM fraction or validation fraction")
        if self.microbatch != "auto" and (not isinstance(self.microbatch, int) or self.microbatch < 1):
            raise ValueError("microbatch must be a positive integer or auto")
        if not self.batch_candidates or min(self.batch_candidates) < 1 or self.separation_radius < 1:
            raise ValueError("Probe batches and separation radius must be positive")
        if self.scheduler not in ("plateau", "cosine", "none"): raise ValueError("Unknown scheduler")
        return self

    def to_dict(self): return asdict(self)


Runtime source: `filament/data.py`

In [ ]:
%%writefile filament_runtime/filament/data.py
"""Dataset integrity, grouped splitting, compact targets and lazy full-image loading."""
import hashlib
import json
import os
import uuid
from threading import Lock
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

from .io import atomic_json, file_hash, fingerprint
from .targets import build_targets, load_image_instances
from .parallel import ordered_map

ANNOTATION_FILENAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
TARGET_VERSION = "independent-coco-separation-8connected-v2"


def find_dataset_root(override: str = "") -> Path:
    """Find existing MAGFiLO layout locally or in nested Kaggle inputs."""
    candidates = [Path(override).expanduser()] if override else []
    if not override:
        for parent in (Path.cwd(), *Path.cwd().parents):
            candidates.extend([parent, parent / "MAGFilo_1.0_Kaggle_2026", parent / "MAGFiLO_1.0_Kaggle_2026"])
        if Path("/kaggle/input").exists():
            candidates.extend(p.parent.parent for p in Path("/kaggle/input").rglob(ANNOTATION_FILENAME))
    for root in candidates:
        if ((root / "train" / ANNOTATION_FILENAME).is_file() and
            (root / "train/train_images").is_dir() and (root / "test/test_images").is_dir()):
            return root.resolve()
    raise FileNotFoundError("Set data_root to the MAGFiLO directory containing train/ and test/")


def discover_test_images(root: Path, expected_size: int = 2048, workers=0) -> list[Path]:
    paths = sorted(p.resolve() for p in (Path(root) / "test/test_images").iterdir()
                   if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    if not paths: raise ValueError("No test images discovered")
    if len({p.stem for p in paths}) != len(paths): raise ValueError("Duplicate test stems")
    def validate(path):
        with Image.open(path) as im:
            if im.size != (expected_size, expected_size): raise ValueError(f"Non-native test image {path}")
    for _ in ordered_map(validate, paths, workers): pass
    return paths


def make_records(root: Path, expected_size: int = 2048, hash_images: bool = True, workers=0) -> tuple[list, dict]:
    """Index annotation records; validate images once and fingerprint decoded pixels.

    Multiple annotators remain separate records. Pixel-identical filenames are
    grouped together for splitting even if JPEG metadata differs. No test labels
    are loaded. Annotation payloads stay in the parent process only after caching.
    """
    root = Path(root)
    annotation_path = root / "train" / ANNOTATION_FILENAME
    data = json.loads(annotation_path.read_text(encoding="utf8"))
    ids = [str(im["id"]) for im in data["images"]]
    if len(set(ids)) != len(ids): raise ValueError("Duplicate COCO image IDs")
    annotation_ids = [str(a["id"]) for a in data["annotations"]]
    if len(set(annotation_ids)) != len(annotation_ids): raise ValueError("Duplicate annotation IDs")
    by_image = defaultdict(list)
    known = set(ids)
    for a in data["annotations"]:
        if str(a["image_id"]) not in known: raise ValueError("Annotation references unknown image")
        if a.get("iscrowd", 0): raise ValueError("Crowd annotation requires an explicit evaluation policy")
        by_image[str(a["image_id"])].append(a)
    files, records = {}, []
    image_dir = (root / "train/train_images").resolve()
    def inspect(im):
        name = im["file_name"]
        path = (image_dir / name).resolve()
        if not path.is_relative_to(image_dir): raise ValueError("Image path escapes dataset")
        shape = (int(im["height"]), int(im["width"]))
        if shape != (expected_size, expected_size): raise ValueError(f"Non-native annotation dimensions: {name}")
        if not path.is_file(): raise FileNotFoundError(path)
        with Image.open(path) as opened:
            if opened.size != shape[::-1]: raise ValueError(f"JPEG and annotation dimensions differ: {name}")
        pixels = read_grayscale(path, shape)
        return name, hashlib.sha256(pixels.tobytes()).hexdigest() if hash_images else name
    unique = {im["file_name"]: im for im in data["images"]}
    files.update(ordered_map(inspect, unique.values(), workers))
    for im in data["images"]:
        name = im["file_name"]
        path = (image_dir / name).resolve()
        if not path.is_relative_to(image_dir): raise ValueError("Image path escapes dataset")
        shape = (int(im["height"]), int(im["width"]))
        if shape != (expected_size, expected_size): raise ValueError(f"Non-native annotation dimensions: {name}")
        records.append(dict(image_id=str(im["id"]), file_name=name, image_path=str(path),
            source_hash=files[name], height=shape[0], width=shape[1], annotations=by_image[str(im["id"])],
            annotation_ids=[str(a["id"]) for a in by_image[str(im["id"])]]))
    groups = group_annotations_by_source(records)
    hashes = defaultdict(list)
    for name, digest in files.items(): hashes[digest].append(name)
    report = dict(annotation_sha256=file_hash(annotation_path), annotation_records=len(records),
        source_images=len(files), annotations=len(annotation_ids),
        multi_annotator_sources={k:len(v) for k,v in groups.items() if len(v)>1},
        duplicate_pixel_sources=[v for v in hashes.values() if len(v)>1],
        instances_per_record=dict(Counter(len(r["annotations"]) for r in records)),
        hash_images=hash_images)
    return records, report


def read_grayscale(path, expected_shape=None) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if im is None: raise OSError(f"Cannot decode image {path}")
    if im.dtype != np.uint8 or im.ndim != 2: raise ValueError("Expected 8-bit grayscale")
    if expected_shape is not None and tuple(im.shape) != tuple(expected_shape): raise ValueError("Image dimensions changed")
    return im


def group_annotations_by_source(records: list) -> dict:
    groups = defaultdict(list)
    for record in records: groups[record["file_name"]].append(record)
    return dict(groups)


def create_train_val_split(records: list, val_fraction=.2, seed=10,
                           multi_annotator_train_only=True) -> dict:
    """Keep prior multi-annotator-in-training policy; group duplicate pixel sources."""
    groups = defaultdict(list)
    for r in records: groups[r.get("source_hash", r["file_name"])].append(r)
    file_groups = group_annotations_by_source(records)
    eligible = sorted(key for key, rows in groups.items() if not multi_annotator_train_only or
        all(len(file_groups[r["file_name"]]) == 1 for r in rows))
    if len(eligible) < 2: raise ValueError("Need at least two eligible independent source groups")
    rng = np.random.default_rng(seed)
    rng.shuffle(eligible)
    desired = max(1, round(val_fraction * len(file_groups)))
    selected, count = set(), 0
    for key in eligible[:-1]:
        if count >= desired: break
        selected.add(key)
        count += len({r["file_name"] for r in groups[key]})
    train, val = [], []
    for key, rows in groups.items(): (val if key in selected else train).extend(rows)
    verify_no_leakage(train, val)
    return dict(train=[r["image_id"] for r in train], val=[r["image_id"] for r in val], seed=seed,
        val_fraction=val_fraction, multi_annotator_train_only=multi_annotator_train_only,
        dataset_fingerprint=record_fingerprint(records))


def record_fingerprint(records: list) -> str:
    return fingerprint([{k:r[k] for k in ("image_id", "file_name", "source_hash", "annotation_ids")}
                        for r in sorted(records, key=lambda r:r["image_id"])])


def verify_no_leakage(train: list, val: list) -> None:
    if not train or not val: raise ValueError("Empty partition")
    for key in ("image_id", "file_name", "source_hash"):
        left = {r.get(key, r["file_name"]) for r in train}
        right = {r.get(key, r["file_name"]) for r in val}
        if left & right: raise ValueError(f"Train/validation leakage by {key}")


def save_split_manifest(path, manifest: dict): atomic_json(path, manifest)


In [ ]:
%%writefile -a filament_runtime/filament/data.py
def load_split_manifest(path, records: list) -> dict:
    result = json.loads(Path(path).read_text(encoding="utf8"))
    if result["dataset_fingerprint"] != record_fingerprint(records): raise ValueError("Split belongs to a different dataset")
    ids = {r["image_id"] for r in records}
    all_ids = result["train"] + result["val"]
    if len(set(all_ids)) != len(all_ids) or set(all_ids) != ids: raise ValueError("Invalid split membership")
    by_id = {r["image_id"]:r for r in records}
    verify_no_leakage([by_id[i] for i in result["train"]], [by_id[i] for i in result["val"]])
    return result


def cache_training_targets(records: list, cache_dir, radius=1, centerline=False, workers=0) -> list:
    """Atomic compressed per-record arrays; original COCO masks remain in compact JSON."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    # Identical annotations may share one target file. Serialize only colliding
    # fingerprints, including concurrent Windows replaces of the same path.
    locks = [Lock() for _ in range(64)]
    def cache_record(r):
        key = fingerprint(dict(version=TARGET_VERSION, shape=[r["height"],r["width"]],
            annotations=r["annotations"], radius=radius, centerline=centerline))
        path = cache_dir / f"{key}.npz"
        original_path = cache_dir / f"{key}.{fingerprint(r['image_id'])}.json"
        with locks[int(key[:8], 16) % len(locks)]:
            if not path.exists():
                targets = build_targets(r, radius, centerline)
                temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
                try:
                    with temporary.open("wb") as handle: np.savez_compressed(handle, **targets)
                    os.replace(temporary, path)
                finally: temporary.unlink(missing_ok=True)
            atomic_json(original_path, dict(image_id=r["image_id"], height=r["height"], width=r["width"],
                annotations=r["annotations"], annotation_ids=r["annotation_ids"]))
        return {k:v for k,v in r.items() if k != "annotations"} | dict(
            target_path=str(path.resolve()), instances_path=str(original_path.resolve()), target_fingerprint=key)
    cached = []
    for index, record in enumerate(ordered_map(cache_record, records, workers)):
        cached.append(record)
        if (index+1) % 100 == 0: print(f"Cached {index+1}/{len(records)} annotation records", flush=True)
    return cached


def cached_instances(record: dict) -> list:
    payload = json.loads(Path(record["instances_path"]).read_text(encoding="utf8"))
    return load_image_instances(payload)


def summarize_dataset(records: list, small_area=20, workers=0) -> tuple[dict, list]:
    """Stream bounded parallel annotation records; retain only compact statistics."""
    def summarize_record(r):
        masks = load_image_instances(r)
        coverage = np.zeros((r["height"], r["width"]), dtype=np.uint32)
        image_areas = []
        for m in masks:
            coverage += m
            image_areas.append(int(m.sum()))
        return dict(image_id=r["image_id"], file_name=r["file_name"], instances=len(masks),
            foreground_ratio=float((coverage>0).mean()), overlap_pixels=int((coverage>1).sum()),
            small_instances=sum(a < small_area for a in image_areas)), image_areas
    rows, areas = [], []
    for index, (row, image_areas) in enumerate(ordered_map(summarize_record, records, workers)):
        rows.append(row); areas.extend(image_areas)
        if (index+1)%100==0: print(f"Validated {index+1}/{len(records)} annotation records",flush=True)
    summary = dict(records=len(rows), instances=len(areas), small_area=small_area,
        small_instances=sum(a<small_area for a in areas), area_quantiles=(
            dict(zip(["min","q25","median","q75","max"],np.quantile(areas,[0,.25,.5,.75,1]).tolist())) if areas else {}),
        overlap_records=sum(r["overlap_pixels"]>0 for r in rows), instance_areas=areas)
    return summary, rows


def normalize_image(image: np.ndarray, mode="unit") -> np.ndarray:
    values = image.astype(np.float32) / 255.
    if mode == "robust":
        lo, hi = np.percentile(values, [1, 99])
        values = np.clip((values-lo)/max(float(hi-lo), 1/255), 0, 1).astype(np.float32)
    elif mode != "unit": raise ValueError("Unknown normalization")
    return values


def augment_sample(image: np.ndarray, targets: dict, rng, options: dict) -> tuple:
    """Same geometry for all maps; mild photometric changes affect input only."""
    arrays = {"image":image, **targets}
    if options.get("flips", True):
        for axis in (0, 1):
            if rng.random() < .5: arrays = {k:np.flip(v, axis) for k,v in arrays.items()}
    if options.get("rotate90", True):
        turns = int(rng.integers(4))
        arrays = {k:np.rot90(v, turns) for k,v in arrays.items()}
    image = arrays.pop("image")
    contrast, brightness = options.get("contrast", .1), options.get("brightness", .05)
    image = np.clip((image-.5)*rng.uniform(1-contrast,1+contrast)+.5+rng.uniform(-brightness,brightness),0,1)
    gamma = options.get("gamma", .1)
    image = image ** rng.uniform(1-gamma,1+gamma)
    if options.get("noise", 0): image += rng.normal(0,options["noise"],image.shape).astype(np.float32)
    return np.ascontiguousarray(np.clip(image,0,1),dtype=np.float32), arrays


class FullImageDataset(Dataset):
    """Lazy native images and cached targets. Augmentation is indexed by epoch/item.

    Shared epoch state also reaches persistent workers. Dense original instances
    are deliberately loaded separately during evaluation, never collated.
    """
    def __init__(self, records: list, augment=False, seed=10, normalization="unit", augment_options=None):
        self.records, self.augment, self.seed = records, augment, seed
        self.normalization, self.options = normalization, augment_options or {}
        self._epoch = torch.zeros((), dtype=torch.int64).share_memory_()

    def set_epoch(self, epoch): self._epoch.fill_(epoch)
    def __len__(self): return len(self.records)

    def __getitem__(self, index):
        r = self.records[index]
        image = normalize_image(read_grayscale(r["image_path"], (r["height"],r["width"])), self.normalization)
        with np.load(r["target_path"], allow_pickle=False) as cached:
            targets = {k:cached[k] for k in ("foreground", "separation", "separation_valid")}
        if self.augment:
            rng = np.random.default_rng(np.random.SeedSequence([self.seed,int(self._epoch),index]))
            image, targets = augment_sample(image, targets, rng, self.options)
        return {"image":torch.from_numpy(np.ascontiguousarray(image[None])),
            **{k:torch.from_numpy(np.ascontiguousarray(v[None])).float() for k,v in targets.items()}, "index":index}


class TrainValidationDataset(Dataset):
    """Share a single persistent worker pool across sequential train/val phases."""
    def __init__(self, train, validation):
        self.train, self.validation = train, validation
        self._validation = torch.zeros((), dtype=torch.int64).share_memory_()

    def select(self, validation): self._validation.fill_(int(validation))
    def __len__(self): return len(self.validation if self._validation.item() else self.train)
    def __getitem__(self, index):
        return (self.validation if self._validation.item() else self.train)[index]


def seed_loader_worker(worker_id):
    cv2.setNumThreads(1)
    torch.set_num_threads(1)


Runtime source: `filament/evaluation.py`

In [ ]:
%%writefile filament_runtime/filament/evaluation.py
"""Organizer-compatible PQ: all IoU>0.5 pairs, global annotator-record aggregation.

No Hungarian assignment: overlapping GT can legitimately match the same prediction
under the supplied notebook. RLE overlap avoids dense N x H x W float stacks.
"""
import numpy as np
import pandas as pd
from pycocotools import mask as mask_utils


def _rles(layers):
    if hasattr(layers, "detach"): layers = layers.detach().cpu().numpy()
    return [mask_utils.encode(np.asfortranarray(np.asarray(m, dtype=np.uint8))) for m in layers]


def get_overlap_matrices(gt_layers, pred_layers) -> tuple[np.ndarray, np.ndarray]:
    """Independent masks -> float64 GT-by-prediction IoU and Dice arrays."""
    labels=getattr(pred_layers,"labels",None)
    if labels is not None:
        # Extracted predictions are disjoint labels, but original GT masks can
        # overlap or contain disconnected regions. Count prediction IDs under
        # each independent GT mask; never build/encode N_pred full-size masks.
        # Counts are integers, preserving the strict organizer IoU>0.5 rule.
        labels=np.asarray(labels)
        count=len(pred_layers)
        areas=np.bincount(labels.ravel(),minlength=count+1)[1:]
        iou=np.zeros((len(gt_layers),count),dtype=np.float64)
        for index,mask in enumerate(gt_layers):
            if hasattr(mask,"detach"): mask=mask.detach().cpu().numpy()
            mask=np.asarray(mask,dtype=bool)
            if mask.shape!=labels.shape: raise ValueError("GT and prediction shapes differ")
            intersection=np.bincount(labels[mask],minlength=count+1)[1:]
            union=int(mask.sum())+areas-intersection
            np.divide(intersection,union,out=iou[index],where=union>0)
        return iou,2*iou/(1+iou)
    gt, pred = _rles(gt_layers), _rles(pred_layers)
    if not gt or not pred:
        return np.zeros((len(gt),len(pred))), np.zeros((len(gt),len(pred)))
    iou = np.asarray(mask_utils.iou(gt, pred, [0]*len(pred)), dtype=np.float64)
    return iou, 2*iou/(1+iou)


def compute_tp_fp_fn(iou_matrix: np.ndarray, threshold=.5) -> dict:
    """Strict > threshold; TP counts pairs, FP/FN count unmatched columns/rows."""
    matrix = np.asarray(iou_matrix)
    if matrix.ndim != 2: raise ValueError("Expected GT x prediction overlap matrix")
    hit = matrix > threshold
    return dict(tp=int(hit.sum()), fp=int((~hit.any(axis=0)).sum()),
        fn=int((~hit.any(axis=1)).sum()), iou_sum=float(matrix[hit].sum()),
        matched_iou=matrix[hit].tolist())


def summarize_validation_results(results: list[dict]) -> dict:
    totals = {k:sum(r[k] for r in results) for k in ("tp","fp","fn","iou_sum")}
    denominator = totals["tp"] + .5*totals["fp"] + .5*totals["fn"]
    return dict(**totals, pq=totals["iou_sum"]/denominator if denominator else 0.,
        matched_instance_iou=totals["iou_sum"]/totals["tp"] if totals["tp"] else 0., records=len(results),
        predicted_instances=sum(r.get("n_pred",0) for r in results),
        fragmented_gt=sum(r.get("fragmented_gt",0) for r in results),
        merged_predictions=sum(r.get("merged_predictions",0) for r in results))


def evaluate_iou_distribution(iou_matrix) -> list:
    matrix = np.asarray(iou_matrix)
    return matrix[matrix > 0].tolist()


def evaluate_dice_distribution(iou_matrix) -> list:
    return [2*x/(1+x) for x in evaluate_iou_distribution(iou_matrix)]


def analyze_fragmentation(iou_matrix) -> dict:
    degrees = (np.asarray(iou_matrix)>0).sum(axis=1)
    return dict(fragmented_gt=int((degrees>1).sum()), gt_overlap_degrees=degrees.tolist())


def analyze_overmerging(iou_matrix) -> dict:
    degrees = (np.asarray(iou_matrix)>0).sum(axis=0)
    return dict(merged_predictions=int((degrees>1).sum()), pred_overlap_degrees=degrees.tolist())


def evaluate_panoptic_quality(gt_instances, pred_instances, threshold=.5) -> dict:
    iou, dice = get_overlap_matrices(gt_instances, pred_instances)
    counts = compute_tp_fp_fn(iou, threshold)
    denominator = counts["tp"]+.5*(counts["fp"]+counts["fn"])
    return dict(**counts, pq=counts["iou_sum"]/denominator if denominator else 0.,
        n_gt=iou.shape[0], n_pred=iou.shape[1],
        iou_distribution=evaluate_iou_distribution(iou), dice_distribution=evaluate_dice_distribution(iou),
        iou_matrix=iou.tolist(), **analyze_fragmentation(iou), **analyze_overmerging(iou))


def get_overlap_df(gt_df: pd.DataFrame, pred_df: pd.DataFrame, shape=(2048,2048)) -> pd.DataFrame:
    """Compatibility with the official CSV helper; score once per annotator-image.

    CSV cannot describe empty annotation records. Production record evaluation
    includes them explicitly (all-empty aggregate scores zero, as official PQ).
    """
    from .submission import decode_rle_to_mask
    gt_keys = gt_df.filament_id.str.rsplit("_",n=1).str[0]
    pred_keys = pred_df.filament_id.str.rsplit("_",n=1).str[0]
    rows=[]
    for key in gt_keys.unique():
        source = key.split("-",1)[1]
        gt = [decode_rle_to_mask(x,shape) for x in gt_df.loc[gt_keys==key,"segmentation_rle"]]
        pred = [decode_rle_to_mask(x,shape) for x in pred_df.loc[pred_keys==source,"segmentation_rle"]]
        iou,dice = get_overlap_matrices(gt,pred)
        rows.append(dict(annotator_image=key,iou_matrix=iou,dice_matrix=dice,n_gt=len(gt),n_pred=len(pred)))
    return pd.DataFrame(rows,columns=["annotator_image","iou_matrix","dice_matrix","n_gt","n_pred"])


def get_pq_score(overlap_df: pd.DataFrame) -> float:
    return summarize_validation_results([compute_tp_fp_fn(r.iou_matrix) for r in overlap_df.itertuples()])["pq"]


Runtime source: `filament/experimental.py`

In [ ]:
%%writefile filament_runtime/filament/experimental.py
"""Native-resolution U-Net worker; launched with torchrun from the full-image notebook.

No tiles or spatial resampling are used. DDP replicates the model on each GPU;
it does not combine the cards into one larger VRAM pool.
"""
import argparse
from contextlib import nullcontext
import csv
import gc
import json
import math
import os
import random
import time
from datetime import timedelta
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.distributed as dist
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset, DistributedSampler


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, dilation=1):
        super().__init__()
        groups = math.gcd(8, out_channels)
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=dilation,
                      dilation=dilation, bias=False),
            nn.GroupNorm(groups, out_channels), nn.ReLU(inplace=False),
            nn.Conv2d(out_channels, out_channels, 3, padding=dilation,
                      dilation=dilation, bias=False),
            nn.GroupNorm(groups, out_channels), nn.ReLU(inplace=False),
        )

    def forward(self, inputs):
        return self.layers(inputs)


class GlobalContext(nn.Module):
    """A pooled image summary makes every output depend on the complete input."""
    def __init__(self, channels):
        super().__init__()
        self.local = DoubleConv(channels, channels, dilation=2)
        self.global_projection = nn.Conv2d(channels, channels, 1)

    def forward(self, inputs):
        local = self.local(inputs)
        return local + self.global_projection(F.adaptive_avg_pool2d(local, 1))


class DecoderStage(nn.Module):
    def __init__(self, in_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, skip_channels, 2, stride=2)
        self.conv = DoubleConv(skip_channels * 2, skip_channels)

    def forward(self, inputs, skip):
        up = self.up(inputs)
        dy, dx = skip.shape[-2]-up.shape[-2], skip.shape[-1]-up.shape[-1]
        up = F.pad(up, [0, dx, 0, dy])
        return self.conv(torch.cat((up, skip), dim=1))


class FullImageUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, base_channels=16,
                 depth=7, max_channels=256, checkpoint_blocks=True):
        super().__init__()
        if min(in_channels, out_channels, base_channels, depth, max_channels) < 1:
            raise ValueError('Channel counts and depth must be positive integers.')
        self.depth = int(depth)
        self.image_size = 2048
        self.checkpoint_blocks = bool(checkpoint_blocks)
        channels = [min(base_channels * 2**index, max_channels)
                    for index in range(self.depth + 1)]
        self.encoders = nn.ModuleList()
        current = in_channels
        for channel in channels[:-1]:
            self.encoders.append(DoubleConv(current, channel))
            current = channel
        self.bottleneck = nn.Sequential(DoubleConv(current, channels[-1]),
                                        GlobalContext(channels[-1]))
        self.decoders = nn.ModuleList()
        current = channels[-1]
        for channel in reversed(channels[:-1]):
            self.decoders.append(DecoderStage(current, channel))
            current = channel
        self.output = nn.Conv2d(current, out_channels, 1)

    def _block(self, block, *inputs):
        if self.checkpoint_blocks and self.training and torch.is_grad_enabled():
            # Checkpointing the complete decoder stage also releases up/cat tensors.
            return checkpoint(block, *inputs, use_reentrant=False)
        return block(*inputs)

    def forward(self, inputs):
        divisor = 2**self.depth
        if inputs.ndim != 4 or min(inputs.shape[-2:]) < divisor:
            raise ValueError(f'Input must be BCHW, with H,W >= {divisor}.')
        skips = []
        for encoder in self.encoders:
            inputs = self._block(encoder, inputs)
            skips.append(inputs)
            inputs = F.max_pool2d(inputs, 2)
        inputs = self._block(self.bottleneck, inputs)
        for decoder, skip in zip(self.decoders, reversed(skips)):
            inputs = self._block(decoder, inputs, skip)
        return self.output(inputs)


Runtime source: `filament/inference.py`

In [ ]:
%%writefile filament_runtime/filament/inference.py
"""Native inference, provenance-bound probability caches and validation searches."""
import itertools
import json
import os
import time
import uuid
from dataclasses import asdict
from contextlib import nullcontext
from pathlib import Path
from queue import Queue
import numpy as np
import pandas as pd
import torch
from .data import read_grayscale, normalize_image, cached_instances, group_annotations_by_source
from .evaluation import evaluate_panoptic_quality, summarize_validation_results
from .io import atomic_json, file_hash, fingerprint
from .postprocess import extract_instances, PostprocessConfig
from .parallel import ordered_map


class MultiDevicePredictor:
    """One model per GPU; CPU jobs borrow an exclusive GPU slot for a forward.

    Call concurrently through ordered_map. Full images are distributed between
    GPUs, so batch one uses both T4s without splitting an image or pooling VRAM.
    """
    def __init__(self, models_and_devices, **predict_options):
        if not models_and_devices: raise ValueError("At least one inference model is required")
        self.slots=Queue()
        for model, device in models_and_devices:
            model.eval()
            self.slots.put((model,torch.device(device)))
        self.predict_options=predict_options

    def __call__(self, path):
        image=read_grayscale(path)
        model,device=self.slots.get()
        try:
            with torch.cuda.device(device) if device.type=="cuda" else nullcontext():
                return predict_full_image(model,image,device,**self.predict_options)
        finally:
            self.slots.put((model,device))


def make_predictor(model, config, device, num_gpus=None):
    """Reuse the loaded model and build a separate replica on each extra GPU."""
    from .model import build_model
    device=torch.device(device)
    models=[(model,device)]
    if device.type=="cuda":
        count=min(2,torch.cuda.device_count()) if num_gpus is None else num_gpus
        if not 1<=count<=torch.cuda.device_count(): raise ValueError("Requested inference GPUs unavailable")
        primary=torch.cuda.current_device() if device.index is None else device.index
        for index in [i for i in range(torch.cuda.device_count()) if i!=primary][:count-1]:
            replica=build_model(config.model).to(torch.device("cuda",index))
            replica.load_state_dict(model.state_dict())
            if config.channels_last: replica.to(memory_format=torch.channels_last)
            models.append((replica,torch.device("cuda",index)))
    return MultiDevicePredictor(models,amp=config.amp,channels_last=config.channels_last,
                                normalization=config.normalization)


def predict_full_image(model, image: np.ndarray, device, amp=True, channels_last=True,
                       normalization="unit", expected_size=2048) -> np.ndarray:
    """One complete native uint8 HW image -> two complete float32 probability maps."""
    if image.dtype!=np.uint8 or image.shape!=(expected_size,expected_size):
        raise ValueError("Prediction requires a complete native uint8 grayscale image")
    device=torch.device(device)
    inputs=torch.from_numpy(normalize_image(image,normalization)[None,None]).to(device,
        **(dict(memory_format=torch.channels_last) if channels_last else {}))
    model.eval()
    with torch.inference_mode(),torch.autocast(device.type,dtype=torch.float16,enabled=amp and device.type=="cuda"):
        logits=model(inputs)
    if logits.shape!=(1,2,expected_size,expected_size): raise ValueError("Unexpected model output shape")
    return logits[0].float().sigmoid().cpu().numpy()


def cache_validation_predictions(model, records, cache_dir, device, provenance, workers=0,
                                 predictor=None, **predict_options) -> dict:
    """One forward pass per source; float16 NPY maps load using mmap during search."""
    namespace=fingerprint(dict(provenance=provenance,predict_options=predict_options))
    directory=Path(cache_dir)/namespace; directory.mkdir(parents=True,exist_ok=True)
    manifest=dict(provenance=provenance,predict_options=predict_options,entries={})
    if predictor is not None and predictor.predict_options!=predict_options:
        raise ValueError("Predictor options must match prediction-cache provenance")
    predictor=predictor or MultiDevicePredictor([(model,device)],**predict_options)
    def cache_source(item):
        name,rows=item
        r=rows[0]
        source_hash=file_hash(r["image_path"])
        path=directory/(fingerprint(dict(name=name,source_hash=source_hash))+".npy")
        started=time.perf_counter()
        cache_hit=path.exists()
        if not cache_hit:
            probabilities=predictor(r["image_path"])
            temporary=path.with_name(path.name+f".{uuid.uuid4().hex}.tmp")
            try:
                with temporary.open("wb") as handle: np.save(handle,probabilities.astype(np.float16),allow_pickle=False)
                os.replace(temporary,path)
            finally: temporary.unlink(missing_ok=True)
        with path.open("rb") as handle:
            array=np.load(handle,allow_pickle=False)
            if array.shape!=(2,r["height"],r["width"]) or not np.isfinite(array).all(): raise ValueError("Corrupt prediction cache")
        return name,dict(path=str(path.resolve()),source_hash=source_hash,
            prediction_seconds=None if cache_hit else time.perf_counter()-started,cache_hit=cache_hit)
    manifest["entries"].update(ordered_map(cache_source,group_annotations_by_source(records).items(),workers))
    atomic_json(directory/"manifest.json",manifest)
    return manifest


def evaluate_cached_predictions(records, manifest, postprocess_config, workers=0) -> tuple[dict,list]:
    def evaluate_source(item):
        name,annotation_records=item
        rows=[]
        entry=manifest["entries"][name]
        if file_hash(annotation_records[0]["image_path"])!=entry["source_hash"]:
            raise ValueError(f"Source image changed after validation caching: {name}")
        probabilities=np.load(entry["path"],mmap_mode="r",allow_pickle=False)
        started=time.perf_counter()
        _,instances=extract_instances(probabilities,postprocess_config)
        seconds=time.perf_counter()-started
        for r in annotation_records:
            result=evaluate_panoptic_quality(cached_instances(r),instances)
            rows.append(dict(**result,image_id=r["image_id"],file_name=name,
                postprocess_seconds=seconds,prediction_seconds=entry["prediction_seconds"]))
        return rows
    rows=[row for source_rows in ordered_map(evaluate_source,group_annotations_by_source(records).items(),workers)
          for row in source_rows]
    return summarize_validation_results(rows),rows


In [ ]:
%%writefile -a filament_runtime/filament/inference.py
def search_postprocessing(records, manifest, grid: dict, output_dir, base_config=None, workers=0) -> tuple[dict,pd.DataFrame]:
    """Tune validation-only postprocessing; predictions are never recomputed."""
    output=Path(output_dir); output.mkdir(parents=True,exist_ok=True)
    base=dict(base_config or {})
    combinations=itertools.product(*(grid[k] for k in grid))
    trials=[]; best=None
    def evaluate_trial(item):
        index,values=item
        cfg=PostprocessConfig(**(base | dict(zip(grid,values))))
        started=time.perf_counter()
        summary,details=evaluate_cached_predictions(records,manifest,asdict(cfg))
        ious=[v for r in details for v in r["iou_distribution"]]
        dices=[v for r in details for v in r["dice_distribution"]]
        row=dict(trial=index,**asdict(cfg),**summary,mean_positive_iou=float(np.mean(ious)) if ious else 0.,
                 mean_positive_dice=float(np.mean(dices)) if dices else 0.,seconds=time.perf_counter()-started)
        return index,cfg,summary,details,row
    # Parallelize trials, not trials AND sources, to respect the total CPU budget.
    for index,cfg,summary,details,row in ordered_map(evaluate_trial,enumerate(combinations),workers):
        trials.append(row)
        pd.DataFrame(trials).to_csv(output/"postprocessing_trials.csv",index=False)
        # Deterministic tie break: first configuration wins.
        if best is None or summary["pq"]>best["validation"]["pq"]:
            best=dict(postprocess=asdict(cfg),validation=summary,trial=index,
                prediction_provenance=manifest["provenance"],validation_image_ids=[r["image_id"] for r in records])
            atomic_json(output/"best_postprocessing.json",best)
            atomic_json(output/"best_validation_details.json",details)
        print(f"Postprocess trial {index}: PQ={summary['pq']:.5f}",flush=True)
    if best is None: raise ValueError("Empty postprocessing grid")
    frame=pd.DataFrame(trials)
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(10,4)); ax.plot(frame.trial,frame.pq,"o-")
    ax.set(xlabel="Validation trial",ylabel="Official-semantics PQ",title="Postprocessing comparison")
    ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(output/"postprocessing_comparison.png",dpi=180); plt.close(fig)
    return best,frame


Runtime source: `filament/io.py`

In [ ]:
%%writefile filament_runtime/filament/io.py
"""Atomic artifact writes; partial files are never treated as complete."""
import hashlib
import json
import os
import uuid
from pathlib import Path


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    try:
        temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf8")
        os.replace(temporary, path)
    finally: temporary.unlink(missing_ok=True)


def fingerprint(value) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def file_hash(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b""): digest.update(chunk)
    return digest.hexdigest()


Runtime source: `filament/losses.py`

In [ ]:
%%writefile filament_runtime/filament/losses.py
"""Reuse the supplied differentiable loss and soft morphology implementation."""
from filament_pq_proxy_loss import (FilamentPQProxyLoss, instance_boundaries,
    soft_erode, soft_open, soft_skeleton, soft_cldice_loss)

__all__ = ["FilamentPQProxyLoss", "instance_boundaries", "soft_erode", "soft_open", "soft_skeleton", "soft_cldice_loss"]


Runtime source: `filament/model.py`

In [ ]:
%%writefile filament_runtime/filament/model.py
"""Single model factory; saved ModifiedUNet is always the default."""
from unet import ModifiedUNet, DoubleConv


def build_model(config: dict):
    options = dict(config)
    name = options.pop("name", "modified")
    if name == "modified": return ModifiedUNet(**options)
    if name == "full_image":
        from .experimental import FullImageUNet
        return FullImageUNet(**options)
    raise ValueError(f"Unknown model {name}")


Runtime source: `filament/parallel.py`

In [ ]:
%%writefile filament_runtime/filament/parallel.py
"""Bounded, ordered CPU work without queuing native-resolution arrays in bulk."""
from collections import deque
from concurrent.futures import ThreadPoolExecutor


def ordered_map(function, items, workers=0):
    """At most `workers` pending jobs; propagate failures and preserve input order.

    OpenCV, NumPy, SciPy and COCO do their expensive work in native code. Threads
    share compact metadata and avoid copying full-resolution masks between processes.
    The serial path is useful for debugging and small synthetic checks.
    """
    if workers < 0:
        raise ValueError("workers cannot be negative")
    if workers <= 1:
        yield from map(function, items)
        return
    iterator = iter(items)
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="filament-cpu") as pool:
        pending = deque()
        for _ in range(workers):
            try:
                pending.append(pool.submit(function, next(iterator)))
            except StopIteration:
                break
        while pending:
            result = pending.popleft().result()
            yield result
            try:
                pending.append(pool.submit(function, next(iterator)))
            except StopIteration:
                pass


Runtime source: `filament/postprocess.py`

In [ ]:
%%writefile filament_runtime/filament/postprocess.py
"""Independent instance extraction methods behind a common probability-map API."""
from dataclasses import dataclass
from collections.abc import Sequence
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed


@dataclass(frozen=True)
class PostprocessConfig:
    method: str = "boundary"
    foreground_threshold: float = .5
    boundary_threshold: float = .5
    min_area: int = 20
    seed_min_area: int = 3
    seed_distance: int = 5
    boundary_weight: float = 2.
    closing_radius: int = 0
    reconnect_distance: int = 0

    def __post_init__(self):
        if self.method not in ("components","boundary","watershed","continuity"): raise ValueError("Unknown extraction method")
        if not 0 < self.foreground_threshold < 1 or not 0 < self.boundary_threshold < 1: raise ValueError("Invalid threshold")
        if min(self.min_area,self.seed_min_area,self.seed_distance) < 1: raise ValueError("Areas and distances must be positive")
        if min(self.boundary_weight,self.closing_radius,self.reconnect_distance) < 0: raise ValueError("Negative morphology parameter")


def component_map(mask: np.ndarray) -> np.ndarray:
    return ndi.label(mask, structure=np.ones((3,3)))[0].astype(np.int32)


def _filter_labels(labels, min_area):
    sizes = np.bincount(labels.ravel())
    keep = sizes >= min_area
    keep[0] = False
    lookup = np.zeros(len(sizes),dtype=np.int32)
    lookup[keep] = np.arange(1,int(keep.sum())+1)
    return lookup[labels]


def _fallback_markers(markers, foreground):
    """An all-separation thin component still receives a seed, so it survives."""
    components = component_map(foreground)
    has_seed = np.unique(components[markers>0])
    missing = np.setdiff1d(np.unique(components), np.r_[0,has_seed])
    next_id = int(markers.max())
    # A full missing component can be one marker, preserving every narrow pixel.
    lookup = np.zeros(int(components.max())+1,dtype=np.int32)
    for key in missing:
        next_id += 1
        lookup[key] = next_id
    return np.where(markers>0,markers,lookup[components]).astype(np.int32)


def connected_components(fg, sep, cfg): return component_map(fg)


def boundary_components(fg, sep, cfg):
    markers = _filter_labels(component_map(fg & (sep < cfg.boundary_threshold)),cfg.seed_min_area)
    markers = _fallback_markers(markers,fg)
    # Geodesic refill allocates separator pixels to instances instead of deleting
    # them. Markers identify disconnected low-separation cores.
    return watershed(np.zeros(fg.shape,np.float32),markers,mask=fg,connectivity=np.ones((3,3))).astype(np.int32)


def marker_watershed(fg, sep, cfg):
    distance = ndi.distance_transform_edt(fg)
    core = fg & (sep < cfg.boundary_threshold)
    peaks = peak_local_max(distance, min_distance=cfg.seed_distance, labels=core.astype(np.uint8), exclude_border=False)
    markers = np.zeros(fg.shape,np.int32)
    if len(peaks): markers[tuple(peaks.T)] = np.arange(1,len(peaks)+1)
    markers = _fallback_markers(markers,fg)
    elevation = -distance + cfg.boundary_weight*sep
    return watershed(elevation,markers,mask=fg,connectivity=np.ones((3,3))).astype(np.int32)


def continuity_refinement(labels, foreground_probability, separation_probability, cfg):
    """Optional conservative gap union; no bridge pixels are added to masks.

    Merge only mutual-nearest, PCA-aligned elongated fragments with a short,
    foreground-supported, low-separation gap. Disconnected predicted regions can
    share an instance ID, just as a disconnected COCO annotation can.
    """
    if cfg.reconnect_distance <= 0: return labels
    from scipy.spatial import cKDTree
    objects = ndi.find_objects(labels)
    endpoints, owners, tangents = [], [], []
    for index, box in enumerate(objects,1):
        if box is None: continue
        ys,xs = np.nonzero(labels[box]==index)
        points = np.column_stack((ys+box[0].start,xs+box[1].start)).astype(float)
        if len(points)<3: continue
        values,vectors = np.linalg.eigh(np.cov(points.T))
        if values[-1] < 4*max(values[0],1): continue
        direction = vectors[:,-1]
        projections = points @ direction
        for i in (int(projections.argmin()),int(projections.argmax())):
            endpoints.append(points[i]); owners.append(index); tangents.append(direction)
    if len(endpoints)<2: return labels
    points=np.asarray(endpoints); tree=cKDTree(points)
    distances, neighbors=tree.query(points,k=min(4,len(points)))
    parent=np.arange(int(labels.max())+1)
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    chosen={}
    for i in range(len(points)):
        candidates=[int(j) for d,j in zip(np.atleast_1d(distances[i])[1:],np.atleast_1d(neighbors[i])[1:])
                    if d<=cfg.reconnect_distance and owners[int(j)]!=owners[i]]
        if candidates: chosen[i]=candidates[0]
    for i,j in chosen.items():
        if i>=j or chosen.get(j)!=i: continue
        delta=points[j]-points[i]; length=np.linalg.norm(delta)
        if length==0: continue
        direction=delta/length
        if min(abs(direction@tangents[i]),abs(direction@tangents[j])) < .95: continue
        line=np.rint(np.linspace(points[i],points[j],int(np.ceil(length))+1)).astype(int)
        ys,xs=line.T
        if (foreground_probability[ys,xs] < cfg.foreground_threshold*.8).any(): continue
        if (separation_probability[ys,xs] >= cfg.boundary_threshold*.5).any(): continue
        parent[find(owners[j])]=find(owners[i])
    lookup=np.array([find(i) for i in range(len(parent))],dtype=np.int32)
    return lookup[labels]


class InstanceMasks(Sequence):
    """Lazy independent binary masks backed by one ID map, bounding RAM usage.

    Iterate or index to materialize a single full-resolution mask. Explicitly
    calling list(...) is possible but can be expensive for fragmented predictions.
    """
    def __init__(self, labels): self.labels=labels; self.count=int(labels.max())
    def __len__(self): return self.count
    def __getitem__(self,index):
        if isinstance(index,slice): return [self[i] for i in range(*index.indices(self.count))]
        if index<0: index+=self.count
        if index<0 or index>=self.count: raise IndexError(index)
        return self.labels==index+1


def extract_instances(probabilities: np.ndarray, config=None) -> tuple[np.ndarray,Sequence]:
    """[2,H,W] probabilities -> ID map and independent HW boolean masks."""
    cfg=config if isinstance(config,PostprocessConfig) else PostprocessConfig(**(config or {}))
    p=np.asarray(probabilities)
    if p.ndim!=3 or p.shape[0]!=2 or not np.isfinite(p).all() or p.min()<0 or p.max()>1:
        raise ValueError("Expected finite [2,H,W] probabilities in [0,1]")
    fg=p[0]>=cfg.foreground_threshold
    if cfg.closing_radius:
        fg=ndi.binary_closing(fg,iterations=cfg.closing_radius) | fg
    if not fg.any(): return np.zeros(fg.shape,np.int32),[]
    method={"components":connected_components,"boundary":boundary_components,
            "watershed":marker_watershed,"continuity":boundary_components}[cfg.method]
    labels=method(fg,p[1],cfg)
    if cfg.method=="continuity": labels=continuity_refinement(labels,p[0],p[1],cfg)
    labels=_filter_labels(labels,cfg.min_area)
    return labels,InstanceMasks(labels) if labels.max() else []


Runtime source: `filament/submission.py`

In [ ]:
%%writefile filament_runtime/filament/submission.py
"""Counts-only COCO RLE CSVs with resumable per-source inference journals."""
import csv
import json
import os
import uuid
from pathlib import Path
import numpy as np
from pycocotools import mask as mask_utils
from .io import atomic_json, fingerprint
from .parallel import ordered_map

COLUMNS = ["filament_id", "segmentation_rle"]


def encode_mask_to_rle(mask: np.ndarray) -> str:
    array=np.asarray(mask)
    if array.ndim!=2 or not np.isin(array,[0,1]).all(): raise ValueError("Expected binary HW mask")
    if not array.any(): raise ValueError("Empty predicted instances must be omitted")
    return mask_utils.encode(np.asfortranarray(array,dtype=np.uint8))["counts"].decode("ascii")


def decode_rle_to_mask(counts: str, shape=(2048,2048)) -> np.ndarray:
    if not isinstance(counts,str) or not counts or any(ord(c)<48 or ord(c)>111 for c in counts):
        raise ValueError("Invalid compressed COCO RLE counts")
    decoded=mask_utils.decode(dict(size=list(shape),counts=counts.encode("ascii")))
    if decoded.shape!=tuple(shape) or not np.isin(decoded,[0,1]).all(): raise ValueError("Invalid decoded RLE")
    # Canonical round trip also catches truncated/overlong run descriptions.
    canonical=mask_utils.encode(np.asfortranarray(decoded))["counts"].decode("ascii")
    if canonical!=counts: raise ValueError("Non-canonical or corrupt RLE")
    return decoded.astype(bool)


def build_filament_id(image_stem: str, index: int) -> str:
    if not image_stem or index<1: raise ValueError("Need original image stem and positive suffix")
    return f"{image_stem}_{index}"


def build_submission_rows(image_stem: str, instances: list, shape=(2048,2048)) -> list[dict]:
    rows=[]
    for i,mask in enumerate(instances,1):
        if mask.shape!=tuple(shape): raise ValueError("Submission mask dimensions differ from original image")
        counts=encode_mask_to_rle(mask)
        if not np.array_equal(decode_rle_to_mask(counts,shape),mask): raise ValueError("RLE round trip failed")
        rows.append(dict(filament_id=build_filament_id(image_stem,i),segmentation_rle=counts))
    return rows


def generate_submission(test_paths: list, predict_instances, output_path, journal_dir,
                        provenance: dict, shape=(2048,2048), workers=0) -> dict:
    """Bounded parallel inference/RLE journals; one deterministic atomic CSV writer.

    predict_instances(path) returns independent bool masks. Journals are bound to
    checkpoint/postprocessing/preprocessing AND test file content signatures by
    the caller. Zero detections have an explicit completed journal with no rows.
    """
    from .io import file_hash
    paths=sorted(map(Path,test_paths))
    if len({p.stem for p in paths})!=len(paths): raise ValueError("Duplicate test stems")
    identity=fingerprint(dict(provenance=provenance,shape=shape))
    journal_dir=Path(journal_dir)/identity
    journal_dir.mkdir(parents=True,exist_ok=True)
    image_counts={}
    output=Path(output_path); output.parent.mkdir(parents=True,exist_ok=True)
    temporary=output.with_name(output.name+f".{uuid.uuid4().hex}.tmp")
    def process_source(p):
        source_hash=file_hash(p)
        journal=journal_dir/(fingerprint(p.stem)+".json")
        result=json.loads(journal.read_text(encoding="utf8")) if journal.exists() else None
        if result is None or result.get("source_hash")!=source_hash:
            rows=build_submission_rows(p.stem,predict_instances(p),shape)
            result=dict(image_stem=p.stem,source_hash=source_hash,rows=rows)
            atomic_json(journal,result)
        if result["image_stem"]!=p.stem: raise ValueError("Journal image mismatch")
        for row in result["rows"]:
            if not decode_rle_to_mask(row["segmentation_rle"],shape).any(): raise ValueError("Empty journal mask")
        return p.stem,result["rows"]
    try:
        with temporary.open("w",newline="",encoding="utf8") as handle:
            writer=csv.DictWriter(handle,fieldnames=COLUMNS); writer.writeheader()
            for stem,rows in ordered_map(process_source,paths,workers):
                writer.writerows(rows)
                image_counts[stem]=len(rows)
        # Validate before making the final artifact visible.
        report=validate_submission(temporary,[p.stem for p in paths],shape,image_counts)
        os.replace(temporary,output)
        atomic_json(output.with_suffix(".manifest.json"),dict(provenance=provenance,
            journal_identity=identity,image_counts=image_counts,shape=list(shape),validation=report))
        return report
    finally: temporary.unlink(missing_ok=True)


def validate_submission(path, image_stems: list, shape=(2048,2048), image_counts=None) -> dict:
    """Stream all rows; no fake rows are permitted for empty images."""
    allowed=set(image_stems)
    if len(allowed)!=len(image_stems): raise ValueError("Duplicate source IDs")
    seen, masks_seen, actual=set(),set(),{s:0 for s in allowed}
    with open(path,newline="",encoding="utf8") as handle:
        reader=csv.DictReader(handle)
        if reader.fieldnames!=COLUMNS: raise ValueError(f"Submission columns must be exactly {COLUMNS}")
        for row in reader:
            if set(row)!=set(COLUMNS) or None in row.values(): raise ValueError("Malformed CSV row")
            fid=row["filament_id"]
            try: stem,suffix=fid.rsplit("_",1)
            except ValueError: raise ValueError("Missing filament suffix") from None
            if stem not in allowed or not suffix.isdigit() or int(suffix)<1: raise ValueError(f"Invalid image ID: {fid}")
            if fid in seen: raise ValueError("Duplicate filament ID")
            seen.add(fid)
            mask=decode_rle_to_mask(row["segmentation_rle"],shape)
            if not mask.any(): raise ValueError("Empty submission mask")
            key=(stem,row["segmentation_rle"])
            if key in masks_seen: raise ValueError("Duplicate instance mask")
            masks_seen.add(key); actual[stem]+=1
    if image_counts is not None and actual!=image_counts: raise ValueError("Missing or unexpected inference records")
    return dict(rows=len(seen),images=len(allowed),zero_detection_images=sum(v==0 for v in actual.values()),
                all_rles_verified=True,coverage_verified=image_counts is not None)


# Original notebook names remain available for callers.
mask_to_rle=encode_mask_to_rle
create_submission=generate_submission


Runtime source: `filament/targets.py`

In [ ]:
%%writefile filament_runtime/filament/targets.py
"""COCO instance identities survive overlaps and disconnected components."""
import numpy as np
from pycocotools import mask as mask_utils
from scipy import ndimage as ndi


def decode_annotation_mask(annotation: dict, height: int, width: int) -> np.ndarray:
    """Decode polygons (x,y), compressed RLE or uncompressed COCO RLE as bool HW."""
    segmentation = annotation.get("segmentation")
    if isinstance(segmentation, list):
        if not segmentation: raise ValueError(f"Empty polygon annotation {annotation.get('id')}")
        for polygon in segmentation:
            points = np.asarray(polygon, dtype=float)
            if points.ndim != 1 or len(points) < 6 or len(points) % 2 or not np.isfinite(points).all():
                raise ValueError(f"Invalid polygon: {annotation.get('id')}")
            xy = points.reshape(-1, 2)
            if (xy < 0).any() or (xy[:, 0] > width).any() or (xy[:, 1] > height).any():
                raise ValueError(f"Polygon outside image: {annotation.get('id')}")
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    elif isinstance(segmentation, dict):
        if list(segmentation.get("size", [])) != [height, width]: raise ValueError("RLE size mismatch")
        rle = dict(segmentation)
        counts = rle.get("counts")
        if isinstance(counts, list):
            if any(not isinstance(c, int) or c < 0 for c in counts) or sum(counts) != height*width:
                raise ValueError("Invalid uncompressed RLE run lengths")
            rle = mask_utils.frPyObjects(rle, height, width)
        elif isinstance(counts, str): rle["counts"] = counts.encode("ascii")
        elif not isinstance(counts, bytes): raise ValueError("Invalid RLE counts")
    else: raise ValueError(f"Missing/invalid segmentation: {annotation.get('id')}")
    result = mask_utils.decode(rle)
    if result.shape != (height, width): raise ValueError("Decoded annotation has incorrect shape")
    result = result.astype(bool)
    if not result.any(): raise ValueError(f"Empty annotation: {annotation.get('id')}; fix the source explicitly")
    return result


def load_image_instances(record: dict) -> list[np.ndarray]:
    """One mask per original annotation; never split by connected components."""
    return [decode_annotation_mask(a, record["height"], record["width"]) for a in record["annotations"]]


def build_foreground_mask(instances: list, shape: tuple) -> np.ndarray:
    result = np.zeros(shape, dtype=bool)
    for mask in instances: result |= mask
    return result


def build_instance_id_map(instances: list, shape: tuple) -> np.ndarray:
    """0 background, positive annotation ordinal, -1 ambiguous overlap."""
    ids = np.zeros(shape, dtype=np.int32)
    for index, mask in enumerate(instances, 1):
        overlap = mask & (ids != 0)
        ids[mask & (ids == 0)] = index
        ids[overlap] = -1
    return ids


def build_boundary_target(instances: list, shape: tuple, radius: int = 1) -> tuple:
    """Inter-instance separation, not exterior contours; return separation and valid.

    Foreground within radius of TWO instances is positive. Overlap and its
    radius-neighborhood are excluded from separation BCE. Union remains supervised.
    """
    if radius < 1: raise ValueError("Separation radius must be positive")
    coverage, nearby = np.zeros(shape, np.uint32), np.zeros(shape, np.uint32)
    for mask in instances:
        coverage += mask
        nearby += ndi.binary_dilation(mask, structure=np.ones((3,3)), iterations=radius)
    ambiguous = coverage > 1
    valid = ~ndi.binary_dilation(ambiguous, structure=np.ones((3,3)), iterations=radius) if ambiguous.any() else ~ambiguous
    return (coverage > 0) & (nearby > 1) & valid, valid


def build_centerline_target(instances: list, shape: tuple) -> np.ndarray:
    from skimage.morphology import skeletonize
    result = np.zeros(shape, dtype=bool)
    for mask in instances: result |= skeletonize(mask)
    return result


def validate_targets(targets: dict, instances: list) -> None:
    shape = targets["foreground"].shape
    for key in ("foreground", "separation", "separation_valid", "instance_ids"):
        if targets[key].shape != shape: raise ValueError(f"Misaligned {key}")
    fg = targets["foreground"].astype(bool)
    if not np.array_equal(fg, build_foreground_mask(instances, shape)): raise ValueError("Incorrect foreground union")
    ids = targets["instance_ids"]
    if not np.array_equal(ids != 0, fg): raise ValueError("IDs lost foreground")
    if (targets["separation"] & ~fg).any(): raise ValueError("Separation outside foreground")
    if (targets["separation"] & ~targets["separation_valid"].astype(bool)).any(): raise ValueError("Ambiguous separation labels")
    if not np.array_equal(ids, build_instance_id_map(instances, shape)): raise ValueError("Instance identities changed")


def build_targets(record: dict, radius=1, centerline=False) -> dict:
    instances = load_image_instances(record)
    shape = (record["height"], record["width"])
    separation, valid = build_boundary_target(instances, shape, radius)
    result = dict(foreground=build_foreground_mask(instances, shape), instance_ids=build_instance_id_map(instances, shape),
                  separation=separation, separation_valid=valid)
    if centerline: result["centerline"] = build_centerline_target(instances, shape)
    validate_targets(result, instances)
    return result


Runtime source: `filament/training.py`

In [ ]:
%%writefile filament_runtime/filament/training.py
"""AMP/DDP training, real-backward memory probes and epoch-boundary recovery."""
import gc
import importlib.metadata
import json
import math
import os
import random
import time
import uuid
from contextlib import nullcontext
from datetime import timedelta
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, Sampler

from .config import Config
from .data import FullImageDataset, TrainValidationDataset, cached_instances, seed_loader_worker
from .evaluation import evaluate_panoptic_quality, summarize_validation_results
from .io import atomic_json
from .losses import FilamentPQProxyLoss
from .model import build_model
from .postprocess import extract_instances

LOSS_NAMES=("loss","bce","dice","boundary","cldice")


class PhaseSampler(Sampler):
    """Switch only after exhausting an epoch iterator; keep one loader worker pool."""
    def __init__(self, train_sampler, validation_indices):
        self.train_sampler, self.validation_indices = train_sampler, validation_indices
        self.validation = False

    def __iter__(self):
        return iter(self.validation_indices if self.validation else self.train_sampler)

    def __len__(self):
        return len(self.validation_indices if self.validation else self.train_sampler)


def seed_everything(seed: int, deterministic=True):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=not deterministic
    # warn_only permits operations without deterministic CUDA kernels to report it.
    torch.use_deterministic_algorithms(deterministic,warn_only=True)


def rng_state() -> dict:
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state() if torch.cuda.is_available() else None)


def restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"]); torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] is not None and torch.cuda.is_available(): torch.cuda.set_rng_state(state["cuda"].cpu())


def _distributed(): return dist.is_available() and dist.is_initialized()


def should_compute_pq(epoch, epochs, pq_every):
    """Always evaluate the first/final epoch, then the requested PQ cadence."""
    return epoch==1 or epoch==epochs or epoch%pq_every==0


def gather_objects(value):
    if not _distributed(): return [value]
    gathered=[None]*dist.get_world_size(); dist.all_gather_object(gathered,value)
    return gathered


def gpu_statistics(device) -> dict:
    if torch.device(device).type!="cuda": return dict(allocated_gib=0.,reserved_gib=0.,peak_allocated_gib=0.,peak_reserved_gib=0.,utilization_percent=None)
    util=None
    try: util=float(torch.cuda.utilization(device))
    except Exception:
        # Optional NVML telemetry must not abort training on an unsupported host.
        pass
    return dict(allocated_gib=torch.cuda.memory_allocated(device)/2**30,
        reserved_gib=torch.cuda.memory_reserved(device)/2**30,
        peak_allocated_gib=torch.cuda.max_memory_allocated(device)/2**30,
        peak_reserved_gib=torch.cuda.max_memory_reserved(device)/2**30,utilization_percent=util)


def _device_batch(batch,device,channels_last):
    # For grayscale N1HW, contiguous(channels_last) is a no-op: it leaves
    # channel stride H*W. cuDNN then returns an NCHW first-convolution gradient,
    # while DDP expects the NHWC parameter stride. `to` explicitly restrides
    # singleton dimensions, and combining it with H2D avoids a second GPU copy.
    return {k:v.to(device,non_blocking=True,**(
        dict(memory_format=torch.channels_last) if k=="image" and channels_last else {}))
        for k,v in batch.items() if k!="index"}


In [ ]:
%%writefile -a filament_runtime/filament/training.py
def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None,
              accumulation_steps=1, amp=True, channels_last=True, grad_clip=1.,
              pq_records=None, postprocess_config=None, log_every=20) -> tuple[dict,list]:
    """Sample-weighted accumulation including short final batches/windows.

    DDP ranks have equal training sampler lengths. Validation bypasses DDP forward
    collectives and uses disjoint non-padding samplers, including empty ranks.
    """
    training=optimizer is not None
    model.train(training)
    forward_model=model if training or not isinstance(model,DDP) else model.module
    device=torch.device(device)
    scaler=scaler or torch.amp.GradScaler("cuda",enabled=amp and device.type=="cuda")
    totals=torch.zeros(len(LOSS_NAMES)+1,dtype=torch.float64,device=device)
    rows=[]; started=time.perf_counter()
    postprocess_seconds=score_seconds=0.
    if training: optimizer.zero_grad(set_to_none=True)
    batches=len(loader)
    batch_size=loader.batch_size
    sample_count=len(loader.sampler)
    if loader.drop_last: sample_count=(sample_count//batch_size)*batch_size
    with (torch.enable_grad() if training else torch.inference_mode()):
        for index,batch in enumerate(loader):
            inputs=_device_batch(batch,device,channels_last)
            count=inputs["image"].shape[0]
            window_start=(index//accumulation_steps)*accumulation_steps
            window_end=min(window_start+accumulation_steps,batches)
            window_samples=min(sample_count,window_end*batch_size)-window_start*batch_size
            sync=index+1==window_end
            context=model.no_sync() if training and isinstance(model,DDP) and not sync else nullcontext()
            with context:
                with torch.autocast(device.type,dtype=torch.float16,enabled=amp and device.type=="cuda"):
                    logits=forward_model(inputs["image"])
                values=criterion(logits,inputs)
                if not torch.isfinite(values["loss"]): raise FloatingPointError("Nonfinite loss")
                if training: scaler.scale(values["loss"]*(count/window_samples)).backward()
            if training and sync:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),grad_clip)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            for i,name in enumerate(LOSS_NAMES): totals[i]+=values[name].detach().double()*count
            totals[-1]+=count
            if not training and pq_records is not None:
                probabilities=logits.float().sigmoid().cpu().numpy()
                for local,record_index in enumerate(batch["index"].tolist()):
                    r=pq_records[record_index]
                    before=time.perf_counter()
                    _,instances=extract_instances(probabilities[local],postprocess_config)
                    after_postprocess=time.perf_counter()
                    row=evaluate_panoptic_quality(cached_instances(r),instances)
                    after_score=time.perf_counter()
                    postprocess_seconds+=after_postprocess-before
                    score_seconds+=after_score-after_postprocess
                    rows.append(dict(**row,image_id=r["image_id"],file_name=r["file_name"],
                        postprocess_evaluation_seconds=time.perf_counter()-before))
            if (index+1)%log_every==0 and (not _distributed() or dist.get_rank()==0):
                print(f"{'Training' if training else 'Validation'} native batch {index+1}/{batches}",flush=True)
            del inputs,logits,values
    if device.type=="cuda": torch.cuda.synchronize(device)
    if _distributed(): dist.all_reduce(totals)
    packed=totals.cpu().tolist(); samples=packed[-1]
    metrics={name:packed[i]/max(samples,1) for i,name in enumerate(LOSS_NAMES)}
    rows=[row for rank_rows in gather_objects(rows) for row in rank_rows]
    if pq_records is not None: metrics.update(summarize_validation_results(rows))
    # Include distributed result gathering and global PQ aggregation in the
    # phase wall time, so validation timings do not hide reporting overhead.
    timings=torch.tensor([time.perf_counter()-started,postprocess_seconds,score_seconds],dtype=torch.float64,device=device)
    if _distributed(): dist.all_reduce(timings,op=dist.ReduceOp.MAX)
    seconds,postprocess_seconds,score_seconds=timings.cpu().tolist()
    metrics.update(samples=samples,seconds=seconds,images_per_second=samples/max(seconds,1e-9))
    if pq_records is not None:
        metrics.update(postprocess_seconds=postprocess_seconds,pq_scoring_seconds=score_seconds)
    return metrics,rows


def train_one_epoch(model,loader,criterion,optimizer,scaler,device,**kwargs):
    return run_epoch(model,loader,criterion,device,optimizer,scaler,**kwargs)[0]


def validate_one_epoch(model,loader,criterion,device,**kwargs):
    return run_epoch(model,loader,criterion,device,**kwargs)


def save_checkpoint(path,model,optimizer,scheduler,scaler,epoch,best_metrics,config,
                    early_stopping,history,rank_rng_states,split_fingerprint):
    """Atomic trusted checkpoint, including rank-local random streams."""
    module=model.module if isinstance(model,DDP) else model
    payload=dict(model=module.state_dict(),optimizer=optimizer.state_dict(),
        scheduler=scheduler.state_dict() if scheduler else None,scaler=scaler.state_dict(),epoch=epoch,
        best_metrics=best_metrics,config=config,early_stopping=early_stopping,history=history,
        rank_rng_states=rank_rng_states,world_size=len(rank_rng_states),split_fingerprint=split_fingerprint)
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+f".{uuid.uuid4().hex}.tmp")
    try: torch.save(payload,temporary); os.replace(temporary,path)
    finally: temporary.unlink(missing_ok=True)


def load_checkpoint(path,model=None,optimizer=None,scheduler=None,scaler=None,map_location="cpu") -> dict:
    # Checkpoints are locally generated, trusted artifacts with numpy RNG tuples.
    state=torch.load(path,map_location=map_location,weights_only=False)
    if model is not None:
        (model.module if isinstance(model,DDP) else model).load_state_dict(state["model"])
    for object_,key in ((optimizer,"optimizer"),(scheduler,"scheduler"),(scaler,"scaler")):
        if object_ is not None and state[key] is not None: object_.load_state_dict(state[key])
    return state


def resume_training(path,model,optimizer,scheduler,scaler,device,rank=0) -> dict:
    state=load_checkpoint(path,model,optimizer,scheduler,scaler,device)
    if rank>=len(state["rank_rng_states"]): raise ValueError("Resume requires the original DDP world size")
    restore_rng(state["rank_rng_states"][rank])
    return state


def update_early_stopping(state: dict, pq: float, patience: int, delta=1e-6) -> tuple[dict,bool,bool]:
    improved=pq>state.get("best_pq",-1)+delta
    result=dict(best_pq=pq if improved else state.get("best_pq",-1),
                stale_evaluations=0 if improved else state.get("stale_evaluations",0)+1)
    return result,improved,result["stale_evaluations"]>=patience


def log_training_metrics(path,history):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_suffix(".tmp")
    pd.DataFrame(history).to_csv(temporary,index=False); os.replace(temporary,path)


In [ ]:
%%writefile -a filament_runtime/filament/training.py
def probe_batch_sizes(model,criterion,sample,config,device) -> tuple[int|None,list]:
    """Actual model + FP32 loss + accumulated backward + AdamW slot allocation.

    Each GPU probes unwrapped, then all ranks agree on a common fit. Model state
    and RNG are restored, so probes never become undocumented training updates.
    """
    device=torch.device(device)
    if device.type!="cuda": return 1,[dict(batch_size=1,fits=True,device="cpu",measured=False)]
    original={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    random_state=rng_state()
    requested=config["microbatch"]
    candidates=sorted(set(config["batch_candidates"] if requested=="auto" else [1,int(requested)]))
    reports=[]
    rank=dist.get_rank() if _distributed() else 0
    try:
        for batch_size in candidates:
            before_probe=time.perf_counter()
            print(f"[rank {rank}] Probing native microbatch={batch_size}...",flush=True)
            optimizer=values=inputs=logits=scaler=None
            model.load_state_dict(original); restore_rng(random_state); model.train()
            gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(device)
            report=dict(batch_size=batch_size,fits=False)
            try:
                optimizer=torch.optim.AdamW(model.parameters(),lr=config["learning_rate"],weight_decay=config["weight_decay"])
                scaler=torch.amp.GradScaler("cuda",enabled=config["amp"],init_scale=1024.)
                batch={k:v.unsqueeze(0).repeat(batch_size,*([1]*v.ndim)) for k,v in sample.items() if k!="index"}
                inputs=_device_batch(batch,device,config["channels_last"])
                torch.cuda.synchronize(device); before=time.perf_counter()
                for step in range(2):
                    optimizer.zero_grad(set_to_none=True)
                    for micro in range(min(config["accumulation_steps"],2)):
                        with torch.autocast("cuda",dtype=torch.float16,enabled=config["amp"]): logits=model(inputs["image"])
                        values=criterion(logits,inputs)
                        scaler.scale(values["loss"]/min(config["accumulation_steps"],2)).backward()
                        logits=values=None
                    scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),config["grad_clip"])
                    scaler.step(optimizer); scaler.update()
                torch.cuda.synchronize(device)
                if not optimizer.state: raise RuntimeError("AMP skipped all probe optimizer steps")
                free,total=torch.cuda.mem_get_info(device)
                peak=torch.cuda.max_memory_allocated(device); reserved=torch.cuda.max_memory_reserved(device)
                margin=max(384*1024**2,sum(p.numel()*p.element_size() for p in model.parameters())*2)
                report.update(fits=peak<total*config["memory_fraction"] and reserved<total*config["memory_fraction"] and free>margin,
                    allocated_bytes=torch.cuda.memory_allocated(device),reserved_bytes=torch.cuda.memory_reserved(device),
                    peak_allocated_bytes=peak,peak_reserved_bytes=reserved,free_bytes=free,total_bytes=total,
                    ddp_margin_bytes=margin,seconds=time.perf_counter()-before,optimizer_state_allocated=True)
            except torch.cuda.OutOfMemoryError:
                report.update(oom=True,peak_allocated_bytes=torch.cuda.max_memory_allocated(device))
            finally:
                model.zero_grad(set_to_none=True); optimizer=values=inputs=logits=scaler=None
                gc.collect(); torch.cuda.empty_cache()
            reports.append(report)
            print(f"[rank {rank}] GPU probe ({time.perf_counter()-before_probe:.1f}s total): {report}",flush=True)
    finally:
        model.load_state_dict(original); restore_rng(random_state); model.zero_grad(set_to_none=True)
    rank_reports=gather_objects(reports)
    feasible=[b for b in candidates if all(next(t for t in r if t["batch_size"]==b)["fits"] for r in rank_reports)]
    selected=(max(feasible) if requested=="auto" else (int(requested) if int(requested) in feasible else 1)) if feasible else None
    return selected,rank_reports


In [ ]:
%%writefile -a filament_runtime/filament/training.py
def train(config: Config, train_records: list, val_records: list, split_fingerprint: str):
    """Entry point for one CUDA process or a torchrun NCCL rank."""
    config.validate()
    rank=int(os.environ.get("RANK",0)); local_rank=int(os.environ.get("LOCAL_RANK",0))
    world=int(os.environ.get("WORLD_SIZE",1)); distributed=world>1
    startup=time.perf_counter()
    def progress(message):
        print(f"[rank {rank} +{time.perf_counter()-startup:.1f}s] {message}",flush=True)
    progress(f"Initializing CUDA device {local_rank}; world_size={world}")
    if not torch.cuda.is_available(): raise RuntimeError("Production training requires CUDA; synthetic unit tests run on CPU")
    if distributed and (os.name=="nt" or not dist.is_nccl_available()): raise RuntimeError("Multi-GPU training requires Linux CUDA/NCCL")
    device=torch.device("cuda",local_rank); torch.cuda.set_device(device)
    torch.set_num_threads(1); cv2.setNumThreads(1)
    seed_everything(config.seed,config.deterministic)
    output=Path(config.output_dir); output.mkdir(parents=True,exist_ok=True)
    progress("CUDA ready")
    if distributed:
        progress("Initializing NCCL process group")
        dist.init_process_group("nccl",timeout=timedelta(minutes=30),device_id=device)
        progress("NCCL ready")
    try:
        previous=load_checkpoint(config.resume) if config.resume else None
        if previous:
            if previous["world_size"]!=world or previous["split_fingerprint"]!=split_fingerprint:
                raise ValueError("Resume world size or split fingerprint differs")
            for key in ("loss","normalization","augment","accumulation_steps","scheduler","separation_radius",
                        "seed","deterministic","centerline","learning_rate","weight_decay","grad_clip","amp"):
                if config.to_dict()[key]!=previous["config"][key]: raise ValueError(f"Resume changes {key}")
            config.model=previous["config"]["model"]
        if config.smoke:
            train_records=train_records[:max(world,config.smoke_records)]
            val_records=val_records[:config.smoke_records]; config.epochs=1
        train_set=FullImageDataset(train_records,True,config.seed,config.normalization,config.augment)
        val_set=FullImageDataset(val_records,False,config.seed,config.normalization)
        criterion=FilamentPQProxyLoss(**config.loss).to(device)
        widths=[config.model.get("init_features",config.model.get("base_channels",16))]
        if not previous and config.model.get("name","modified")=="modified":
            widths+=sorted({w for w in config.width_fallbacks if w<widths[0]},reverse=True)
        model=None; all_probes=[]
        for width in widths:
            progress(f"Building model width={width}; loading probe sample")
            if model is not None: del model; gc.collect(); torch.cuda.empty_cache()
            if config.model.get("name","modified")=="modified": config.model["init_features"]=width
            model=build_model(config.model).to(device)
            if config.channels_last: model.to(memory_format=torch.channels_last)
            microbatch,reports=probe_batch_sizes(model,criterion,train_set[0],config.to_dict(),device)
            all_probes.append(dict(width=width,ranks=reports))
            if microbatch is not None: break
        if microbatch is None: raise RuntimeError("No native-resolution training batch fits; reduce channel width or free GPU memory")
        config.microbatch=microbatch
        progress(f"Selected microbatch={microbatch} per GPU; effective batch={microbatch*world*config.accumulation_steps}")
        progress("Wrapping model with DDP" if distributed else "Setting up single-GPU training")
        model=DDP(model,device_ids=[local_rank],broadcast_buffers=False,gradient_as_bucket_view=True) if distributed else model
        sampler=DistributedSampler(train_set,num_replicas=world,rank=rank,seed=config.seed,drop_last=False)
        workers=config.workers//world+int(rank<config.workers%world)
        options=dict(batch_size=microbatch,pin_memory=True,worker_init_fn=seed_loader_worker)
        worker_options=dict(num_workers=workers)
        if workers: worker_options.update(prefetch_factor=config.prefetch_factor,persistent_workers=True,multiprocessing_context="spawn")
        generator=torch.Generator()
        phase_dataset=TrainValidationDataset(train_set,val_set)
        phase_sampler=PhaseSampler(sampler,list(range(rank,len(val_set),world)))
        loader=DataLoader(phase_dataset,sampler=phase_sampler,generator=generator,**options,**worker_options)
        progress(f"Loader ready: {workers} CPU workers on this rank; prefetch={config.prefetch_factor}")
        optimizer=torch.optim.AdamW(model.parameters(),lr=config.learning_rate,weight_decay=config.weight_decay)
        scaler=torch.amp.GradScaler("cuda",enabled=config.amp)
        scheduler=(torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",factor=.5,patience=2) if config.scheduler=="plateau"
            else torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,config.epochs) if config.scheduler=="cosine" else None)
        start=0; history=[]; early={}; best={}; elapsed_before=0.
        if previous:
            previous=resume_training(config.resume,model,optimizer,scheduler,scaler,device,rank)
            start=previous["epoch"]; history=previous["history"]; early=previous["early_stopping"]; best=previous["best_metrics"]
            elapsed_before=history[-1].get("total_training_seconds",0) if history else 0
            del previous
        hardware=gather_objects(dict(rank=rank,name=torch.cuda.get_device_name(device),cpu_workers=workers,
            vram_gib=torch.cuda.get_device_properties(device).total_memory/2**30))
        if rank==0:
            atomic_json(output/"effective_config.json",config.to_dict())
            versions={}
            for package in ("numpy","scipy","scikit-image","pandas","matplotlib","pillow","pycocotools","opencv-python","opencv-python-headless"):
                try: versions[package]=importlib.metadata.version(package)
                except importlib.metadata.PackageNotFoundError: pass
            atomic_json(output/"runtime_report.json",dict(hardware=hardware,probes=all_probes,world_size=world,
                torch_version=torch.__version__,cuda_version=torch.version.cuda,package_versions=versions))
        started=time.perf_counter()
        for epoch in range(start+1,config.epochs+1):
            progress(f"Starting epoch {epoch}/{config.epochs}")
            train_set.set_epoch(epoch); generator.manual_seed(config.seed+epoch+rank)
            sampler.set_epoch(epoch)
            phase_dataset.select(False); phase_sampler.validation=False
            torch.cuda.reset_peak_memory_stats(device)
            train_metrics=train_one_epoch(model,loader,criterion,optimizer,scaler,device,
                accumulation_steps=config.accumulation_steps,amp=config.amp,channels_last=config.channels_last,grad_clip=config.grad_clip,log_every=config.log_every)
            do_pq=should_compute_pq(epoch,config.epochs,config.pq_every)
            if rank==0:
                print(f"Epoch {epoch} training finished: {train_metrics['seconds']:.1f}s; "
                      f"starting validation (PQ={'on' if do_pq else 'off'})",flush=True)
            phase_dataset.select(True); phase_sampler.validation=True
            val_metrics,details=validate_one_epoch(model,loader,criterion,device,
                amp=config.amp,channels_last=config.channels_last,pq_records=val_records if do_pq else None,
                postprocess_config=config.postprocess,log_every=config.log_every)
            if scheduler:
                scheduler.step(val_metrics["loss"]) if config.scheduler=="plateau" else scheduler.step()
            improved=stop=False
            if do_pq:
                early,improved,stop=update_early_stopping(early,val_metrics["pq"],config.patience)
                if improved: best=dict(epoch=epoch,**val_metrics)
            memory=gather_objects(gpu_statistics(device))
            row=dict(epoch=epoch,**{f"train_{k}":v for k,v in train_metrics.items()},
                **{f"val_{k}":v for k,v in val_metrics.items()},learning_rate=optimizer.param_groups[0]["lr"],
                total_training_seconds=elapsed_before+time.perf_counter()-started,
                peak_allocated_gib=max(m["peak_allocated_gib"] for m in memory),
                peak_reserved_gib=max(m["peak_reserved_gib"] for m in memory),gpu_statistics=memory)
            history.append(row)
            random_states=gather_objects(rng_state())
            if rank==0:
                kwargs=dict(model=model,optimizer=optimizer,scheduler=scheduler,scaler=scaler,epoch=epoch,
                    best_metrics=best,config=config.to_dict(),early_stopping=early,history=history,
                    rank_rng_states=random_states,split_fingerprint=split_fingerprint)
                if improved: save_checkpoint(output/"checkpoints/best.pt",**kwargs)
                save_checkpoint(output/"checkpoints/last.pt",**kwargs)
                log_training_metrics(output/"logs/history.csv",history)
                if do_pq: atomic_json(output/f"validation/epoch_{epoch:03d}.json",dict(metrics=val_metrics,records=details))
                print(f"Epoch {epoch}: loss={train_metrics['loss']:.4f}, val={val_metrics['loss']:.4f}, "
                    f"PQ={val_metrics.get('pq','skipped')}, train={train_metrics['seconds']:.1f}s, "
                    f"validation={val_metrics['seconds']:.1f}s, "
                    f"postprocess={val_metrics.get('postprocess_seconds',0.):.1f}s, "
                    f"PQ scoring={val_metrics.get('pq_scoring_seconds',0.):.1f}s",flush=True)
            if distributed: dist.barrier()
            if stop: break
    except torch.cuda.OutOfMemoryError:
        atomic_json(output/f"oom_rank_{rank}.json",dict(rank=rank,microbatch=config.microbatch,model=config.model))
        raise
    finally:
        if distributed and dist.is_initialized(): dist.destroy_process_group()


Runtime source: `filament/visualization.py`

In [ ]:
%%writefile filament_runtime/filament/visualization.py
"""Readable complete-disk and native-pixel detail plots saved as PNG artifacts."""
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from .data import read_grayscale,cached_instances
from .postprocess import extract_instances


def _save(fig,path):
    if path:
        path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
        fig.savefig(path,dpi=180,bbox_inches="tight")
    return fig


def _overlay(ax,image,masks,title):
    rgb=np.repeat(image[...,None],3,axis=-1)
    colors=plt.get_cmap("turbo")
    for i,mask in enumerate(masks):
        color=np.asarray(colors((i*.61803398875)%1))[:3]*255
        rgb[mask]=(rgb[mask].astype(np.float32)*.4+color*.6).astype(np.uint8)
    # One composited RGB layer avoids retaining N full-resolution RGBA layers.
    ax.imshow(rgb,interpolation="nearest")
    ax.set_title(title); ax.axis("off")


def _zoom_box(masks,shape,size=256):
    union=np.zeros(shape,bool)
    for mask in masks: union|=mask
    ys,xs=np.where(union)
    cy,cx=(int(np.median(ys)),int(np.median(xs))) if len(ys) else (shape[0]//2,shape[1]//2)
    if len(ys):
        nearest=int(np.argmin((ys-cy)**2+(xs-cx)**2))
        cy,cx=int(ys[nearest]),int(xs[nearest])
    y=max(0,min(cy-size//2,max(0,shape[0]-size)))
    x=max(0,min(cx-size//2,max(0,shape[1]-size)))
    return slice(y,y+size),slice(x,x+size)


def plot_dataset_inspection(record,path=None):
    image=read_grayscale(record["image_path"]); masks=cached_instances(record)
    with np.load(record["target_path"]) as target:
        fg,sep,valid=(target[k] for k in ("foreground","separation","separation_valid"))
    fig,axes=plt.subplots(2,4,figsize=(17,9),layout="constrained")
    _overlay(axes[0,0],image,[],"Original H-alpha")
    _overlay(axes[0,1],image,masks,"Independent annotations (overlaps retained)")
    axes[0,2].imshow(fg,cmap="gray"); axes[0,2].set_title("Foreground union")
    axes[0,3].imshow(sep,cmap="magma",vmin=0,vmax=1); axes[0,3].set_title("Inter-instance separation")
    zoom=_zoom_box(masks,image.shape)
    _overlay(axes[1,0],image[zoom],[m[zoom] for m in masks],"Native pixels: annotation detail")
    for ax,data,title in zip(axes[1,1:],(fg[zoom],sep[zoom],~valid[zoom]),("Foreground detail","Separation detail","Excluded overlap neighborhood")):
        ax.imshow(data,cmap="gray",interpolation="nearest"); ax.set_title(title)
    for ax in axes.flat: ax.axis("off")
    fig.suptitle(record["file_name"],fontsize=15)
    return _save(fig,path)


def plot_training(history_path,output_dir):
    frame=pd.read_csv(history_path)
    fig,axes=plt.subplots(2,3,figsize=(15,8),layout="constrained")
    for split in ("train","val"): axes[0,0].plot(frame.epoch,frame[f"{split}_loss"],"o-",markersize=3,label=split)
    axes[0,0].set_title("Composite loss"); axes[0,0].legend()
    for name in ("bce","dice","boundary","cldice"):
        axes[0,1].plot(frame.epoch,frame[f"train_{name}"],"o-",markersize=3,label=name)
        axes[0,2].plot(frame.epoch,frame[f"val_{name}"],"o-",markersize=3,label=name)
    axes[0,1].set_title("Training terms"); axes[0,2].set_title("Validation terms")
    axes[0,1].legend(); axes[0,2].legend()
    axes[1,0].plot(frame.epoch,frame.learning_rate,"o-",markersize=3); axes[1,0].set_title("Learning rate")
    for col in ("peak_allocated_gib","peak_reserved_gib"): axes[1,1].plot(frame.epoch,frame[col],"o-",markersize=3,label=col)
    axes[1,1].set_title("GPU memory (GiB)"); axes[1,1].legend(fontsize=8)
    if "val_pq" in frame: axes[1,2].plot(frame.epoch,frame.val_pq,"o-")
    axes[1,2].set_title("Instance PQ used for model selection")
    for ax in axes.flat: ax.set_xlabel("Epoch"); ax.grid(alpha=.25)
    return _save(fig,Path(output_dir)/"training.png")


def plot_prediction(record,probabilities,postprocess_config,path=None,title=None):
    image=read_grayscale(record["image_path"]); gt=cached_instances(record)
    labels,pred=extract_instances(probabilities,postprocess_config)
    fig,axes=plt.subplots(2,6,figsize=(23,8),layout="constrained")
    zoom=_zoom_box(gt or pred,image.shape)
    for row,region in enumerate(((slice(None),slice(None)),zoom)):
        _overlay(axes[row,0],image[region],[],"Original" if row==0 else "Native-pixel detail")
        for col,channel,name in ((1,0,"Foreground probability"),(2,1,"Separation probability")):
            axes[row,col].imshow(probabilities[channel][region],cmap="magma",vmin=0,vmax=1,interpolation="nearest")
            axes[row,col].set_title(name)
        axes[row,3].imshow(labels[region]>0,cmap="gray",interpolation="nearest"); axes[row,3].set_title("Retained foreground")
        _overlay(axes[row,4],image[region],(m[region] for m in pred),f"Predicted instances ({len(pred)})")
        _overlay(axes[row,5],image[region],[m[region] for m in gt],f"GT instances ({len(gt)})")
    for ax in axes.flat: ax.axis("off")
    fig.suptitle(title or record["file_name"])
    return _save(fig,path)


def plot_distributions(details,path=None):
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax,key,title in zip(axes[:2],("iou_distribution","dice_distribution"),("Positive-pair IoU","Positive-pair Dice")):
        values=[x for r in details for x in r[key]]
        weights=np.ones(len(values))/max(len(values),1)
        ax.hist(values,bins=np.linspace(0,1,51),weights=weights,color="navy",alpha=.7)
        ax.set(title=title,xlim=(0,1),ylabel="Probability",xlabel="Score")
    for key,label,offset in (("gt_overlap_degrees","GT 1:n",.2),("pred_overlap_degrees","Pred n:1",-.2)):
        degrees=[d for r in details for d in r[key]]
        bins=np.bincount(degrees) if degrees else np.zeros(1)
        axes[2].bar(np.arange(len(bins))+offset,bins,width=.4,label=label)
    axes[2].set(title="Any-overlap relations",xlabel="Overlap degree",ylabel="Instances"); axes[2].legend()
    return _save(fig,path)


def plot_error_gallery(records,manifest,details,postprocess_config,output_dir):
    if not details: return []
    by_id={r["image_id"]:r for r in records}
    choices={"best":max(details,key=lambda r:r["pq"]),"worst":min(details,key=lambda r:r["pq"])}
    for name,key in (("false_positives","fp"),("missed","fn"),("fragmentation","fragmented_gt"),("overmerging","merged_predictions")):
        example=max(details,key=lambda r:r[key])
        if example[key]>0: choices[name]=example
    outputs=[]
    for name,result in choices.items():
        record=by_id[result["image_id"]]
        probabilities=np.load(manifest["entries"][record["file_name"]]["path"],mmap_mode="r")
        path=Path(output_dir)/f"{name}.png"
        fig=plot_prediction(record,probabilities,postprocess_config,path,
            f"{name}: PQ={result['pq']:.3f}, FP={result['fp']}, FN={result['fn']}")
        plt.close(fig); outputs.append(str(path))
    return outputs


Runtime source: `filament/workflow.py`

In [ ]:
%%writefile filament_runtime/filament/workflow.py
"""Reproducible preparation and safe single/DDP process orchestration."""
import json
import os
import subprocess
import sys
import time
import uuid
from pathlib import Path

import pandas as pd
import torch
from .config import Config
from .data import (find_dataset_root,make_records,create_train_val_split,load_split_manifest,
    save_split_manifest,cache_training_targets,summarize_dataset,verify_no_leakage,TARGET_VERSION,discover_test_images)
from .io import atomic_json, fingerprint


def prepare_experiment(config: Config, statistics=True, indexed=None) -> dict:
    config.validate()
    root=find_dataset_root(config.data_root)
    config.data_root=str(root)
    output=Path(config.output_dir).resolve(); output.mkdir(parents=True,exist_ok=True)
    config.output_dir=str(output)
    records,integrity=indexed if indexed is not None else make_records(root,config.image_size,workers=config.workers)
    integrity["test_source_images"]=len(discover_test_images(root,config.image_size,workers=config.workers))
    atomic_json(output/"data/integrity.json",integrity)
    split_path=output/"split_manifest.json"
    manifest=load_split_manifest(split_path,records) if split_path.exists() else create_train_val_split(
        records,config.val_fraction,config.seed,config.multi_annotator_train_only)
    if not split_path.exists(): save_split_manifest(split_path,manifest)
    # Annotation content as well as image identities bind the prepared runtime.
    selected=records
    if config.smoke:
        by_id={r["image_id"]:r for r in records}
        selected=[by_id[i] for i in manifest["train"][:config.smoke_records]+manifest["val"][:config.smoke_records]]
    data_identity=fingerprint(dict(split=manifest,annotations=integrity["annotation_sha256"],
        radius=config.separation_radius,centerline=config.centerline,
        selected_ids=[r["image_id"] for r in selected],smoke=config.smoke,target_version=TARGET_VERSION))
    if statistics:
        summary,rows=summarize_dataset(selected,workers=config.workers)
        atomic_json(output/"data/statistics.json",summary)
        pd.DataFrame(rows).to_csv(output/"data/statistics_per_record.csv",index=False)
    cached=cache_training_targets(selected,output/"targets",config.separation_radius,config.centerline,workers=config.workers)
    by_id={r["image_id"]:r for r in cached}
    train_records=[by_id[i] for i in manifest["train"] if i in by_id]
    val_records=[by_id[i] for i in manifest["val"] if i in by_id]
    verify_no_leakage(train_records,val_records)
    runtime=dict(config=config.to_dict(),train_records=train_records,val_records=val_records,
                 split_fingerprint=data_identity,smoke=config.smoke)
    atomic_json(output/"prepared.json",runtime)
    atomic_json(output/"config.json",config.to_dict())
    return runtime


def launch_training(runtime: dict, num_gpus=None) -> None:
    """Launch worker module; whole-job OOM fallback restarts all DDP ranks safely.

    Each failed epoch is replayed from last.pt. Smaller microbatch is automatic;
    resolution never changes. Architecture fallback is only allowed before any
    checkpoint exists. torchrun terminates peer ranks if one rank fails.
    """
    config=Config(**runtime["config"])
    available=torch.cuda.device_count()
    count=min(2,available) if num_gpus is None else int(num_gpus)
    if count<1 or count>available: raise ValueError("Requested GPU count unavailable")
    if count>1 and os.name=="nt": raise RuntimeError("NCCL DDP requires Linux; select one GPU on Windows")
    output=Path(config.output_dir); output.mkdir(parents=True,exist_ok=True)
    source_root=Path(__file__).resolve().parent.parent
    environment=os.environ.copy()
    environment.update(PYTHONUNBUFFERED="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",OPENBLAS_NUM_THREADS="1",
                       CUBLAS_WORKSPACE_CONFIG=":4096:8")
    environment["PYTHONPATH"]=str(source_root)+os.pathsep+environment.get("PYTHONPATH","")
    attempt=0
    while True:
        for stale in output.glob("oom_rank_*.json"): stale.unlink()
        run_file=output/"worker_runtime.json"
        runtime["config"]=config.to_dict(); atomic_json(run_file,runtime)
        command=[sys.executable,"-u"]
        if count>1:
            # This launcher is single-host. Pin both rendezvous and the address
            # advertised to workers to loopback instead of resolving the host FQDN.
            command += ["-m","torch.distributed.run","--nnodes=1","--rdzv_backend=c10d",
                "--rdzv_endpoint=127.0.0.1:0","--local_addr=127.0.0.1",f"--rdzv_id={uuid.uuid4().hex}",
                f"--nproc_per_node={count}"]
        command += ["-m","filament.cli","worker","--runtime",str(run_file)]
        log=output/"logs"/f"training_attempt_{attempt}.log"; log.parent.mkdir(parents=True,exist_ok=True)
        print(f"Launching {count} GPU rank(s), {config.workers} total CPU workers; log: {log}",flush=True)
        launched=time.perf_counter()
        with log.open("w",encoding="utf8") as handle:
            process=subprocess.Popen(command,cwd=source_root,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                text=True,bufsize=1,start_new_session=os.name!="nt",
                creationflags=subprocess.CREATE_NO_WINDOW if os.name=="nt" else 0)
            try:
                for line in process.stdout:
                    print(line,end="",flush=True); handle.write(line); handle.flush()
                returncode=process.wait()
            except BaseException:
                if os.name!="nt":
                    import signal
                    os.killpg(process.pid,signal.SIGTERM)
                else:
                    subprocess.run(["taskkill","/PID",str(process.pid),"/T","/F"],capture_output=True)
                process.wait(); raise
        if returncode==0:
            print(f"Training workers completed in {time.perf_counter()-launched:.1f}s",flush=True)
            return
        signals=list(output.glob("oom_rank_*.json"))
        if not signals: raise RuntimeError(f"Training worker failed ({returncode}); inspect {log}")
        failures=[json.loads(p.read_text(encoding="utf8")) for p in signals]
        old_batch=min(int(r["microbatch"]) for r in failures)
        last=output/"checkpoints/last.pt"
        if old_batch>1:
            config.microbatch=max(1,old_batch//2)
            if last.exists(): config.resume=str(last)
        elif not last.exists() and not config.resume and config.model.get("name","modified")=="modified":
            old_width=failures[0]["model"]["init_features"]
            options=[w for w in config.width_fallbacks if w<old_width]
            if not options: raise RuntimeError(f"Native microbatch one exhausted width fallbacks; inspect {log}")
            config.model["init_features"]=max(options); config.microbatch=1
        else: raise RuntimeError(f"Native microbatch one cannot recover without changing a saved architecture; inspect {log}")
        attempt+=1
        print(f"Restarting all ranks after CUDA OOM with microbatch={config.microbatch}",flush=True)


In [ ]:
sys.path.insert(0, str(SOURCE_DIR.resolve()))
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt

print(
    "Python:",
    sys.version,
    "| PyTorch:",
    torch.__version__,
    "| CUDA:",
    torch.version.cuda,
)
for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(i, gpu.name, f"{gpu.total_memory / 2**30:.2f} GiB")
if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a Kaggle GPU accelerator or use a CUDA-enabled local environment."
    )
torch.set_num_threads(1)
import cv2

cv2.setNumThreads(1)

## 5. Imports and configuration

All major choices live in `Config`. Width 16, GroupNorm, non-reentrant activation
checkpointing, FP16, microbatch probing, and accumulation are conservative starting
points for T4s. Microbatches 1 and 2 are probed within each 15 GiB device's
memory limit; the supplied dual-T4 log confirms 2 fits and 4 OOMs. Width 8 is an
automatic pre-training fallback; image resolution
never changes. Four workers are a **total** CPU budget: two loader processes per
T4 during training/validation; four CPU threads for preparation, search and RLE.
Both GPUs also run validation/test inference. Native-library threads are capped
at one per worker. Plotting, reductions and final file assembly remain ordered.


In [ ]:
from filament.config import Config
from filament.io import atomic_json, file_hash
from filament.data import *
from filament.targets import *
from filament.model import build_model
from filament.losses import FilamentPQProxyLoss
from filament.workflow import prepare_experiment, launch_training
from filament.training import load_checkpoint, seed_everything
from filament.inference import *
from filament.evaluation import *
from filament.postprocess import extract_instances
from filament.submission import *
from filament.visualization import *

MODE = "full"  # 'smoke' or 'full'
CPU_WORKERS = min(32, os.cpu_count() or 1)
REQUESTED_GPUS = 2
PQ_EVERY = 2  # PQ on first/every second/final epoch; loss every epoch.
DATA_ROOT_OVERRIDE = ""
RUN_TRAINING = True
RUN_TEST_INFERENCE = MODE == "full"
ON_KAGGLE = (
    os.name != "nt"
    and Path("/kaggle/working").exists()
    and Path("/kaggle/input").exists()
)
OUTPUT_DIR = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd()) / f"filament_{MODE}"
cfg = Config(
    data_root=DATA_ROOT_OVERRIDE,
    output_dir=str(OUTPUT_DIR),
    smoke=MODE == "smoke",
    workers=CPU_WORKERS,
    prefetch_factor=2,
    deterministic=False,
)
# deterministic=False enables cuDNN autotuning for fixed native image shapes.
# Set it True for deterministic kernels, at a potential throughput cost.
if MODE == "smoke":
    cfg.epochs = 1
    cfg.model["init_features"] = 8
    cfg.microbatch = 1
# Resume a complete epoch: cfg.resume = str(OUTPUT_DIR / 'checkpoints/last.pt')
cfg.pq_every = PQ_EVERY
cfg.validate()
seed_everything(cfg.seed, cfg.deterministic)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(cfg.to_dict())

## 6. Dataset discovery

Find `train/train_images`, the original MAGFiLO annotation JSON, and
`test/test_images`. Inspect native dimensions, image IDs, missing files and
pixel-identical duplicates. Test image pixels are used only during final inference.


In [ ]:
DATA_ROOT = find_dataset_root(cfg.data_root)
records, integrity = make_records(DATA_ROOT, workers=cfg.workers)
integrity["test_source_images"] = len(
    discover_test_images(DATA_ROOT, workers=cfg.workers)
)  # Headers/inventory only; no test tuning.
print({k: v for k, v in integrity.items() if k != "multi_annotator_sources"})
print("Multi-annotator source count:", len(integrity["multi_annotator_sources"]))
atomic_json(OUTPUT_DIR / "data/integrity.json", integrity)

## 7. Annotation inspection

One COCO image record identifies one annotator and source. Multiple polygons in
one annotation are decoded as one independent instance using pycocotools. Empty
annotation lists are valid images with zero instances; a malformed or empty
individual annotation raises an error and is never silently removed.


In [ ]:
example = records[0]
instances = load_image_instances(example)
print(
    example["image_id"], example["file_name"], "independent instances:", len(instances)
)
print("Native dimensions:", example["height"], example["width"])
del instances

## 8. Dataset statistics

Statistics use a bounded four-worker pool. Smoke mode inspects a small sample; full
mode validates every original annotation. Area histograms, foreground fraction,
overlap pixels and small-instance counts are saved without retaining dense masks.


In [ ]:
stats_records = records[:8] if cfg.smoke else records
statistics, statistics_rows = summarize_dataset(stats_records, workers=cfg.workers)
atomic_json(OUTPUT_DIR / "data/statistics.json", statistics)
pd.DataFrame(statistics_rows).to_csv(
    OUTPUT_DIR / "data/statistics_per_record.csv", index=False
)
display(pd.DataFrame(statistics_rows).head())
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist([r["instances"] for r in statistics_rows], bins=20)
axes[0].set_title("Instances per annotator record")
axes[1].hist(statistics["instance_areas"], bins=40)
axes[1].set(xscale="log", title="Instance areas (native pixels)")
axes[2].hist([r["foreground_ratio"] for r in statistics_rows], bins=20)
axes[2].set_title("Foreground fraction")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "data/statistics.png", dpi=180)
plt.show()

## 9. Train/validation splitting

Multi-annotator source files stay in training by default. Validation size targets
20% of unique source filenames and is drawn only from eligible single-annotator
groups. Duplicate pixel groups remain intact. The manifest binds image identities
and is reused rather than resampled. This policy facilitates baseline comparison
but validation does not represent annotator disagreement; grouped cross-validation
including multiple annotators is a recommended subsequent experiment.


In [ ]:
split_path = OUTPUT_DIR / "split_manifest.json"
split = (
    load_split_manifest(split_path, records)
    if split_path.exists()
    else create_train_val_split(
        records, cfg.val_fraction, cfg.seed, cfg.multi_annotator_train_only
    )
)
save_split_manifest(split_path, split)
by_id = {r["image_id"]: r for r in records}
verify_no_leakage([by_id[i] for i in split["train"]], [by_id[i] for i in split["val"]])
print(
    "Training records:", len(split["train"]), "| Validation records:", len(split["val"])
)

## 10. Target generation and validation

Foreground is the union of all independent masks. IDs are 0 for background,
positive for unambiguous annotation membership, and -1 for overlap. Separation is
foreground near **two different annotations**, not the contour of one filament.
Overlap and a radius neighborhood are excluded from separation BCE. Foreground
overlap remains supervised; originals are retained in compact COCO JSON for PQ.
Preparation writes compressed atomic per-record arrays with content fingerprints.
```text
Annotation polygons / RLE → one independent mask per annotation
                       ├→ union → foreground
                       ├→ distinct-mask proximity → separation
                       ├→ overlapping coverage → exclusion mask / ID = -1
                       └→ originals retained → instance PQ
```


In [ ]:
runtime = prepare_experiment(cfg, statistics=False, indexed=(records, integrity))
train_records, val_records = runtime["train_records"], runtime["val_records"]
figure = plot_dataset_inspection(val_records[0], OUTPUT_DIR / "figures/targets.png")
plt.show()
plt.close(figure)
print("Prepared:", len(train_records), len(val_records))

## 11. Full-resolution preprocessing

Read native uint8 grayscale JPEG; divide by 255 to float32. Optional `robust`
mode clips/normalizes the 1st–99th intensity percentiles consistently in train,
validation and test. No resizing or tiling occurs. Skip alignment pads only a
rounding difference when odd test dimensions are used in synthetic checks.


In [ ]:
native_image = read_grayscale(val_records[0]["image_path"], (2048, 2048))
normalized = normalize_image(native_image, cfg.normalization)
assert normalized.shape == (2048, 2048) and normalized.dtype == np.float32
print(normalized.shape, float(normalized.min()), float(normalized.max()))
del native_image, normalized

## 12. Augmentation

Flips and 90-degree rotations are shared across every target map. Mild contrast,
brightness, gamma and optional noise affect only the input. The RNG is indexed by
seed, epoch and record; validation/test are deterministic. Native morphology is
preserved without interpolation or arbitrary-angle resampling.


In [ ]:
preview_dataset = FullImageDataset(
    train_records, True, cfg.seed, cfg.normalization, cfg.augment
)
preview_dataset.set_epoch(1)
sample = preview_dataset[0]
print({k: tuple(v.shape) for k, v in sample.items() if hasattr(v, "shape")})
del sample, preview_dataset

## 13. Dataset and DataLoader

Lazy loading returns input and three B1HW supervision maps, not an all-dataset
instance stack. Training DDP uses `DistributedSampler.set_epoch`; validation
partitions have no padding duplicates. Pinned memory, configurable prefetch and
persistent workers are enabled only when workers > 0. Shared epoch state keeps
persistent-worker augmentations correct. One persistent pool serves both phases,
so four workers are not doubled during validation. Training sampler padding can repeat a
small number of records to keep rank lengths equal.


In [ ]:
validation_dataset = FullImageDataset(val_records, normalization=cfg.normalization)
print(
    "Input/targets are loaded on demand:", len(validation_dataset), "validation records"
)

## 14. Custom U-Net architecture

```text
1 × 2048² → f × 2048² ───────────────────────────────┐
             ↓ maxpool                             │ skip
           2f × 1024² ───────────────────────┐       │
             ↓                              │       │
           4f × 512² ────────────────┐       │       │
             ↓                      │       │       │
           8f × 256² ───────┐       │       │       │
             ↓              │       │       │       │
          16f × 128² → transpose conv + concat → DoubleConv (reverse levels)
                                                     ↓
                                              1×1 conv → 2 × 2048²
                                                 foreground / separation
```

This preserves the saved `ModifiedUNet` architecture, `_match_and_concat`, and
raw logits. GroupNorm supports batch one. BatchNorm is available only with
checkpointing disabled to avoid recomputation updating running statistics twice.
The previous deeper `FullImageUNet` remains opt-in via `name='full_image'`, with
`base_channels`, `depth`, `max_channels`, and `checkpoint_blocks` parameters.


In [ ]:
probe_model = build_model(
    dict(name="modified", init_features=2, checkpoint_blocks=False)
)
with torch.inference_mode():
    for height, width in [(32, 48), (33, 35)]:
        assert probe_model(torch.zeros(1, 1, height, width)).shape == (
            1,
            2,
            height,
            width,
        )
print("Even/odd two-channel shape checks passed")
del probe_model

## 15. Loss functions and mathematics

$$L=w_b\,BCEWithLogits(F,Y)+w_d\left(1-{2\sum\sigma(F)Y+1\over\sum\sigma(F)+\sum Y+1}\right)
+w_s\,BCE_{posweight}(S,T_s;V)+w_c\,clDice.$$

$V$ excludes ambiguous overlap neighborhoods. Separation BCE is averaged over
valid pixels **per image**, then over images. Foreground BCE and soft Dice retain
all union pixels. FP32 arithmetic is enforced even under AMP. Each term is logged.
The supplied soft erosion/opening/skeleton/clDice functions are reused. clDice is
disabled by default because native-resolution skeleton graphs consume substantial
VRAM. This objective is a surrogate, not a differentiable official PQ.
```text
Foreground logits → BCE + soft Dice (+ optional clDice) ─┐
Separation logits + target + valid mask → weighted BCE ──┤
                         configurable weighted sum ←───┘
                                     ↓
                            AMP-scaled backward
```


In [ ]:
small_model = build_model(dict(name="modified", init_features=2))
logits = small_model(torch.randn(1, 1, 33, 35))
target = {
    "foreground": torch.zeros(1, 1, 33, 35),
    "separation": torch.zeros(1, 1, 33, 35),
    "separation_valid": torch.ones(1, 1, 33, 35),
}
terms = FilamentPQProxyLoss(**cfg.loss)(logits, target)
terms["loss"].backward()
assert torch.isfinite(terms["loss"])
print({k: float(v.detach()) for k, v in terms.items()})
del small_model, logits, target, terms

## 16. GPU setup and memory probing

Each worker probes complete native images with the configured loss, accumulated
backward, AMP and **AdamW state allocation**. Allocated/reserved/peak memory and
DDP headroom are measured. Probes run before DDP wrapping, so an OOM can be handled
without trapping other ranks in a collective. Weights and RNG are restored.
All GPUs must pass a common microbatch. Width fallback is explicit in runtime logs.

```text
Notebook → torchrun → rank 0/T4 → full-image batch → gradients ─┐
                   → rank 1/T4 → full-image batch → gradients ─┤ all-reduce
                                                        AdamW update on each rank
```

VRAM is not pooled across GPUs. Unexpected training OOM triggers a whole-job
restart from the last complete epoch using a smaller microbatch. Width changes
are permitted only before a checkpoint exists. No automatic spatial reduction.


In [ ]:
NUM_GPUS = min(REQUESTED_GPUS, torch.cuda.device_count())
if os.name == "nt":
    NUM_GPUS = 1
print("Worker GPU count:", NUM_GPUS)
if NUM_GPUS < REQUESTED_GPUS:
    print(
        "Using available/supported GPUs; select GPU T4 x2 in Kaggle settings for both T4s."
    )
print(
    "Total CPU workers:",
    cfg.workers,
    "| workers per GPU:",
    [
        cfg.workers // NUM_GPUS + int(rank < cfg.workers % NUM_GPUS)
        for rank in range(NUM_GPUS)
    ],
)
print(
    "Nominal effective batch = selected microbatch ×",
    NUM_GPUS,
    "×",
    cfg.accumulation_steps,
)

## 17. Training configuration

AdamW, clipping, AMP scaling, plateau/cosine/no scheduler, exact short-window
accumulation, early stopping by PQ evaluations, and best/last atomic checkpoints
are configured centrally. Resume requires the same split, loss, preprocessing,
augmentation, accumulation, scheduler and GPU count. The effective model width
and batch are persisted with checkpoints. Epoch RNG state is stored per rank.


In [ ]:
atomic_json(OUTPUT_DIR / "requested_config.json", cfg.to_dict())
print(
    "Epochs:",
    cfg.epochs,
    "| PQ frequency:",
    cfg.pq_every,
    "| clDice:",
    cfg.loss["cldice_weight"],
)

## 18. Training execution

An independent generated worker process avoids notebook DDP spawning issues.
Logs stream to this cell and disk. Interrupting training leaves the last complete
epoch intact; set `cfg.resume` in section 5 and rerun preparation/training.
Rank-labelled startup timings identify import, CUDA, NCCL, probe and loader waits.
Single-host torchrun uses an explicit loopback rendezvous. Smoke mode verifies
execution, not model quality. Each epoch logs separate training/validation wall
times and maximum per-rank CPU postprocessing/PQ-scoring times. Validation loss
runs every epoch; `PQ_EVERY=2` evaluates instance PQ on the first, every second and
final epoch. Set it to 1 for every-epoch checkpoint selection. Early-stopping
patience counts PQ evaluations. All strict IoU>0.5 pairs still count, including
overlapping GT; disjoint prediction IDs accelerate overlap counting on the CPU.


In [ ]:
if RUN_TRAINING:
    # Avoid retaining parent-process CUDA models while workers train.
    import gc

    gc.collect()
    torch.cuda.empty_cache()
    launch_training(runtime, NUM_GPUS)
else:
    print("Using existing checkpoint; ensure OUTPUT_DIR is the intended experiment.")

## 19. Training visualization

Loss terms, validation PQ, learning rate, epoch/total duration and per-rank GPU
statistics are persisted. Utilization is sampled where NVML is available; it is
not an epoch-average measurement. The training graph distinguishes diagnostics
from the instance objective used to select the best checkpoint.


In [ ]:
history = pd.read_csv(OUTPUT_DIR / "logs/history.csv")
display(history.tail())
fig = plot_training(OUTPUT_DIR / "logs/history.csv", OUTPUT_DIR / "figures")
plt.show()
plt.close(fig)

## 20. Checkpoint loading

Use the checkpoint's **effective** architecture and preprocessing after probing,
not the requested width. Only load trusted locally generated PyTorch checkpoints.
The best checkpoint is selected by strict improvement in global validation PQ.


In [ ]:
CHECKPOINT = OUTPUT_DIR / "checkpoints/best.pt"
state = load_checkpoint(CHECKPOINT)
effective_cfg = Config(**state["config"])
DEVICE = torch.device("cuda:0")
model = build_model(effective_cfg.model).to(DEVICE)
model.load_state_dict(state["model"])
model.eval()
if effective_cfg.channels_last:
    model.to(memory_format=torch.channels_last)
predictor = make_predictor(model, effective_cfg, DEVICE, NUM_GPUS)
print(
    "Selected epoch:",
    state["epoch"],
    "| validation PQ:",
    state["best_metrics"].get("pq"),
)
del state
PROVENANCE = {
    "checkpoint_sha256": file_hash(CHECKPOINT),
    "normalization": effective_cfg.normalization,
    "split_fingerprint": runtime["split_fingerprint"],
}

## 21. Full-resolution validation inference

Run one forward pass per unique validation filename across both GPUs and cache
two float16 maps. Four bounded CPU jobs overlap image reads, GPU work and writes.
Cache keys include checkpoint, normalization, inference options and source file
hash. Search reads maps with memory mapping and never repeats model inference.
Inference during training uses FP32 probabilities; search uses float16 caches,
which can shift pixels extremely close to a threshold. This precision tradeoff is
recorded and should be checked in a final experiment if consequential.


In [ ]:
prediction_manifest = cache_validation_predictions(
    model,
    val_records,
    OUTPUT_DIR / "validation_predictions",
    DEVICE,
    PROVENANCE,
    workers=cfg.workers,
    predictor=predictor,
    amp=effective_cfg.amp,
    channels_last=effective_cfg.channels_last,
    normalization=effective_cfg.normalization,
)
print("Cached source predictions:", len(prediction_manifest["entries"]))

## 22. Instance extraction and postprocessing

Compare components, separation-aware connected cores with geodesic refill, and
marker watershed using distance peaks and a separation elevation term. Every
foreground component gets a fallback marker even if entirely classified as a
separator. Small-instance filtering happens after refill. Optional continuity
refinement joins only short, mutually nearest PCA-aligned, probability-supported
gaps; it is disabled by default and may still merge distinct filaments.

```text
Foreground threshold → optional closing → seed extraction ← separation
 → component / geodesic refill / marker watershed → min area → IDs → independent masks
```


In [ ]:
baseline_summary, baseline_details = evaluate_cached_predictions(
    val_records, prediction_manifest, effective_cfg.postprocess, workers=cfg.workers
)
print(baseline_summary)

## 23. Official PQ evaluation

Global aggregation retains every annotator record and every independent mask.
The repository tests compare matrices, grouping, strict threshold, empty cases,
overlapping annotations and final score with functions extracted unchanged from
the supplied self-evaluation notebook. The record API additionally includes empty
GT records that cannot be represented in an instance-only CSV.
```text
Each original annotator record + source predictions → GT × prediction overlaps
 → every IoU > 0.5 pair: TP / IoU sum; unmatched columns: FP; unmatched rows: FN
 → sum across annotator records → global PQ ratio
```


In [ ]:
synthetic = np.zeros((16, 16), bool)
synthetic[2:12, 3:7] = True
assert evaluate_panoptic_quality([synthetic, synthetic], [synthetic])["tp"] == 2
assert evaluate_panoptic_quality([synthetic], [synthetic])["pq"] == 1
assert evaluate_panoptic_quality([], [])["pq"] == 0
atomic_json(
    OUTPUT_DIR / "validation/baseline.json",
    {"summary": baseline_summary, "records": baseline_details},
)
print("Global PQ:", baseline_summary["pq"])

## 24. Dice, IoU and fragmentation analysis

Positive-pair distributions and IoU>0 degree counts reproduce the reference
analysis. Matched IoU is reported separately from the positive-pair mean. Tiny
incidental overlaps also contribute to fragmentation/merging degrees, so inspect
the spatial examples before diagnosing an error from a count alone.


In [ ]:
fig = plot_distributions(
    baseline_details, OUTPUT_DIR / "figures/overlap_distributions.png"
)
plt.show()
plt.close(fig)
display(
    pd.DataFrame(
        [
            {
                k: r[k]
                for k in [
                    "file_name",
                    "pq",
                    "tp",
                    "fp",
                    "fn",
                    "fragmented_gt",
                    "merged_predictions",
                    "n_pred",
                ]
            }
            for r in baseline_details
        ]
    )
    .sort_values("pq")
    .head()
)

## 25. Validation-based postprocessing optimization

No retraining: reuse cached maps. First configuration wins ties. Save all trials
to CSV and the winning parameters/provenance to JSON. Four CPU workers evaluate
trials concurrently with bounded memory and deterministic tie order. This holdout
is used for model and threshold selection, so its score is selection-biased;
reserve another grouped holdout or use cross-validation for a final estimate.


In [ ]:
SEARCH_GRID = {
    "method": ["components", "boundary", "watershed"],
    "foreground_threshold": [0.4, 0.5, 0.6],
    "boundary_threshold": [0.4, 0.6],
    "min_area": [10, 20],
}
if cfg.smoke:
    SEARCH_GRID = {"method": ["components", "boundary"], "foreground_threshold": [0.5]}
# Optional grid entries: seed_distance, seed_min_area, boundary_weight,
# closing_radius, reconnect_distance (only for method='continuity').
best_postprocessing, trials = search_postprocessing(
    val_records,
    prediction_manifest,
    SEARCH_GRID,
    OUTPUT_DIR / "search",
    effective_cfg.postprocess,
    workers=cfg.workers,
)
POSTPROCESS = best_postprocessing["postprocess"]
display(trials.sort_values("pq", ascending=False).head())

## 26. Qualitative prediction review

Save full disks and native-pixel zooms for best/worst PQ, false positives, missed
filaments, fragmentation and merging when those error cases exist. Crops are
display-only. The probability maps always come from complete-image inference.


In [ ]:
selected_summary, selected_details = evaluate_cached_predictions(
    val_records, prediction_manifest, POSTPROCESS, workers=cfg.workers
)
gallery = plot_error_gallery(
    val_records,
    prediction_manifest,
    selected_details,
    POSTPROCESS,
    OUTPUT_DIR / "figures/errors",
)
review = val_records[0]
probabilities = np.load(
    prediction_manifest["entries"][review["file_name"]]["path"], mmap_mode="r"
)
fig = plot_prediction(
    review, probabilities, POSTPROCESS, OUTPUT_DIR / "figures/prediction_review.png"
)
plt.show()
plt.close(fig)
print("Saved review figures:", gallery)

## 27. Test inference

Use the selected checkpoint and validation-selected parameters. Each original
test image receives one full-resolution forward pass, distributed across both
T4s. Four CPU jobs overlap reading, GPU forwards, postprocessing and RLE encoding.
Per-image atomic journals
support resuming an interruption and record zero detections explicitly. Journals
are bound to checkpoint, postprocessing, normalization and source file content.


In [ ]:
if RUN_TEST_INFERENCE and effective_cfg.smoke:
    raise ValueError("Smoke checkpoints cannot create a final competition submission.")
test_paths = (
    discover_test_images(DATA_ROOT, workers=cfg.workers) if RUN_TEST_INFERENCE else []
)


def predict_test_instances(path):
    probabilities = predictor(path)
    return extract_instances(probabilities.astype(np.float16), POSTPROCESS)[1]


print("Test sources to process:", len(test_paths))

## 28. Submission generation

One row per non-empty predicted filament: `filament_id,segmentation_rle`.
IDs retain the original test stem and add `_1`, `_2`, … . Values are compressed
COCO RLE counts only, encoded from Fortran-order 2048-square masks. Standard CSV
escaping is used; no quotation characters are inserted into values. Zero detected
filaments means no row, plus an explicit completed entry in the sidecar manifest.

```text
native independent masks → Fortran-order uint8 → pycocotools encode → counts string
 → original_stem_suffix → per-image journal → atomic submission.csv
```


In [ ]:
if RUN_TEST_INFERENCE:
    submission_report = generate_submission(
        test_paths,
        predict_test_instances,
        OUTPUT_DIR / "submission.csv",
        OUTPUT_DIR / "inference_journal",
        PROVENANCE | {"postprocess": POSTPROCESS},
        workers=cfg.workers,
    )
    print(submission_report)
else:
    print("Final submission is disabled in smoke mode.")

## 29. Submission validation

Verify every RLE by decode/re-encode, exact schema, unique IDs, original stem
mapping, binary non-empty masks, duplicate masks, and complete source accounting
against the inference manifest. Shape is fixed to 2048 × 2048. Never add fabricated
empty-instance rows for missing detections. The standalone synthetic check below
also exercises the RLE path in smoke mode.


In [ ]:
rle_test = np.zeros((32, 32), bool)
rle_test[4:22, 8:13] = True
assert np.array_equal(
    decode_rle_to_mask(encode_mask_to_rle(rle_test), (32, 32)), rle_test
)
if RUN_TEST_INFERENCE:
    import json

    sidecar = json.loads((OUTPUT_DIR / "submission.manifest.json").read_text())
    print(
        validate_submission(
            OUTPUT_DIR / "submission.csv",
            [p.stem for p in test_paths],
            image_counts=sidecar["image_counts"],
        )
    )
else:
    print("Synthetic COCO RLE round trip passed; no final CSV created in smoke mode.")

## 30. Results summary, reproducibility and limitations

Inspect the selected checkpoint, `effective_config.json`, split, runtime report,
epoch logs, target fingerprints, validation reports, search CSV/JSON and inference
manifest. Archive them together. Full training-dependent artifacts exist only
after the relevant successful stage; smoke outputs are separate.

Limitations: two pixel heads do not encode overlapping instance membership;
watershed can fragment long structures; conservative reconnection can overmerge;
the default single-annotator validation policy underrepresents disagreement;
postprocessing search uses float16 maps; no temporal/date split is assumed;
source grouping prevents exact duplicate leakage but not all correlated dates.
Two-T4 NCCL performance and competition quality must be measured in Kaggle.

Recommended controlled experiments: width 8/16/32 after real memory probing,
separation radius, sparse-positive weighting, watershed seed spacing, grouped
cross-validation including multiple annotators, and only then optional clDice.
Never use test ground truth or public-leaderboard selection for these choices.


In [ ]:
summary = {
    "mode": MODE,
    "checkpoint": str(CHECKPOINT),
    "checkpoint_sha256": file_hash(CHECKPOINT),
    "validation": selected_summary,
    "postprocess": POSTPROCESS,
    "test_submission_created": RUN_TEST_INFERENCE,
    "warning": (
        "Smoke validation is an execution check, not a performance estimate."
        if cfg.smoke
        else "Validation reused for model/postprocess selection; confirm on an untouched grouped holdout."
    ),
}
atomic_json(OUTPUT_DIR / "results_summary.json", summary)
print(summary)